# 02 - Inyecci?n de Anomal?as Sint?ticas full_v3

Versi?n completa con todas las anomal?as y `Sensor Atascado` integrado desde la versi?n final v25/v25f.

**Salida:** `data/interim/02_datos_inyectados_full_v3.parquet`


In [ ]:
# --- Intel Extension for Scikit-learn ----------------------------------------
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# --- LibrerÃ­as estÃ¡ndar -------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings, hashlib
import matplotlib.ticker as mticker

# --- Scikit-learn (ya parcheado si Intel Extension estÃ¡ disponible) -----------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# --- ConfiguraciÃ³n del proyecto -----------------------------------------------
import sys
PROJECT_DIR = os.getcwd()
if not os.path.exists(os.path.join(PROJECT_DIR, 'config.py')):
    raise FileNotFoundError(
        f"No se encuentra config.py en el directorio actual: {PROJECT_DIR}. "
        "Abre y ejecuta el notebook desde la carpeta del proyecto."
    )
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import config as project_config
from config import *
from config_full_v3 import (
    DATOS_FALTANTES_MAX_GAP_MIN_FULL_V3,
    DATOS_FALTANTES_ANCHOR_MAX_GAP_MIN_FULL_V3,
    DATOS_FALTANTES_DELTA_MAX_FULL_V3,
    DATOS_FALTANTES_REAL_MAX_TRAIN_FULL_V3,
    DATOS_FALTANTES_REAL_BALANCE_SEED_FULL_V3,
    DATOS_FALTANTES_REAL_MAX_POR_COMBO_FULL_V3,
)
print(f"Config cargado desde: {project_config.__file__}")

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA - algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")

# --- Rutas aisladas full_v3 ---------------------------------------------------
PARQUET_02_FULL_V3 = os.path.join(DATA_INTERIM, '02_datos_inyectados_full_v3.parquet')
PARQUET_02 = PARQUET_02_FULL_V3
OUT_EVENTOS_STUCK_FULL_V3 = os.path.join(DATA_RESULTS, '02_stuck_full_v3_eventos.csv')
OUT_AUDITORIA_STUCK_FULL_V3 = os.path.join(DATA_RESULTS, '02_stuck_full_v3_auditoria.csv')
print(f'Salida full_v3: {PARQUET_02_FULL_V3}')


## 0. Cargar datos del paso anterior

In [ ]:
# Cargar el dataset limpio generado en 01_carga_datos.ipynb
df_trabajo = pd.read_parquet(PARQUET_01)
print(f"Dataset cargado: {df_trabajo.shape}")
print(f"Periodo: {df_trabajo['Fecha'].min()} -> {df_trabajo['Fecha'].max()}")

# Seguridad: no permitir que filas usar_en_train=False entren en inyeccion.
if COLUMNA_USAR_EN_TRAIN in df_trabajo.columns:
    filas_totales = len(df_trabajo)
    mask_usable = df_trabajo[COLUMNA_USAR_EN_TRAIN].fillna(False).astype(bool)
    df_trabajo = df_trabajo.loc[mask_usable].reset_index(drop=True)
    df_trabajo = df_trabajo.drop(columns=[COLUMNA_USAR_EN_TRAIN])
    print(f"Filas conservadas para inyeccion: {len(df_trabajo):,} / {filas_totales:,}")
    print(f"Columna '{COLUMNA_USAR_EN_TRAIN}' eliminada del dataframe de trabajo tras el filtrado.")
else:
    print(f"Advertencia: columna '{COLUMNA_USAR_EN_TRAIN}' no disponible; se usaran todas las filas.")

# Seguridad adicional: si quedase algun verano en el parquet, lo excluimos aqui tambien.
mask_sin_cultivo = pd.Series(False, index=df_trabajo.index)
for inicio, fin in PERIODOS_SIN_CULTIVO:
    mask_sin_cultivo |= (df_trabajo['Fecha'] >= inicio) & (df_trabajo['Fecha'] <= fin)
if mask_sin_cultivo.any():
    df_trabajo = df_trabajo.loc[~mask_sin_cultivo].reset_index(drop=True)
    print(f"Filas extra excluidas por periodo sin cultivo: {int(mask_sin_cultivo.sum()):,}")

df_trabajo['Fecha'] = pd.to_datetime(df_trabajo['Fecha'])

df_original = df_trabajo.copy()   # copia sin modificar para referencias estadisticas

# Metadatos de trazabilidad de la inyeccion. No son features del modelo.
df_trabajo['sensor_anomalia'] = pd.NA
df_trabajo['anomalia_id'] = pd.NA
df_trabajo['season_anomalia'] = pd.NA
df_trabajo['period_anomalia'] = pd.NA

# ---------------------------------------------------------------------------
# Etiquetado directo de NaNs reales antes de inyectar anomal?as sint?ticas.
# Un NaN operativo no debe entrar al entrenamiento como normal:
#   - hueco corto/suave/reconstruible -> Datos Faltantes real
#   - hueco largo/brusco/sin anclas -> excluido del entrenamiento
# Implementaci?n por segmentos para que el cuaderno no se bloquee en datasets grandes.
# ---------------------------------------------------------------------------
import time as _time_nan_real

_t0_nan_real = _time_nan_real.time()

if COLUMNA_USAR_EN_TRAIN not in df_trabajo.columns:
    df_trabajo[COLUMNA_USAR_EN_TRAIN] = True
if 'fuente_anomalia' not in df_trabajo.columns:
    df_trabajo['fuente_anomalia'] = pd.NA

SENSORES_NAN_REAL = [s for s in COLUMNAS_SENSORES if s in df_original.columns]
MAX_GAP_MIN_NAN_REAL = DATOS_FALTANTES_MAX_GAP_MIN_FULL_V3
ANCHOR_MAX_GAP_MIN_NAN_REAL = DATOS_FALTANTES_ANCHOR_MAX_GAP_MIN_FULL_V3
DELTA_MAX_NAN_REAL = DATOS_FALTANTES_DELTA_MAX_FULL_V3

fechas_nan_real = pd.to_datetime(df_original['Fecha'])
fechas_ns_nan_real = fechas_nan_real.astype('int64').to_numpy()
_diffs_min_nan_real = fechas_nan_real.diff().dropna().dt.total_seconds().div(60.0)
STEP_MIN_NAN_REAL = float(_diffs_min_nan_real.median()) if len(_diffs_min_nan_real) else 0.5
if not np.isfinite(STEP_MIN_NAN_REAL) or STEP_MIN_NAN_REAL <= 0:
    STEP_MIN_NAN_REAL = 0.5

n_nan_real = len(df_original)
nan_real_arr = np.zeros((n_nan_real, len(SENSORES_NAN_REAL)), dtype=bool)
recon_real_arr = np.zeros((n_nan_real, len(SENSORES_NAN_REAL)), dtype=bool)

for j, sensor_nan in enumerate(SENSORES_NAN_REAL):
    valores_nan = pd.to_numeric(df_original[sensor_nan], errors='coerce').to_numpy(dtype=float)
    mask_nan_sensor = np.isnan(valores_nan)
    nan_real_arr[:, j] = mask_nan_sensor
    if not mask_nan_sensor.any():
        continue

    valid_idx = np.flatnonzero(~mask_nan_sensor)
    if len(valid_idx) == 0:
        continue

    padded_nan = np.r_[False, mask_nan_sensor, False]
    cambios_nan = np.flatnonzero(padded_nan[1:] != padded_nan[:-1])
    starts_nan = cambios_nan[::2]
    ends_nan = cambios_nan[1::2] - 1
    delta_max_sensor = float(DELTA_MAX_NAN_REAL.get(sensor_nan, np.inf))

    for start_nan, end_nan in zip(starts_nan, ends_nan):
        n_puntos_nan = int(end_nan - start_nan + 1)
        dur_min_nan = ((fechas_ns_nan_real[end_nan] - fechas_ns_nan_real[start_nan]) / 1e9 / 60.0) + STEP_MIN_NAN_REAL
        if dur_min_nan > MAX_GAP_MIN_NAN_REAL:
            continue

        pos_prev = np.searchsorted(valid_idx, start_nan) - 1
        pos_next = np.searchsorted(valid_idx, end_nan + 1)
        if pos_prev < 0 or pos_next >= len(valid_idx):
            continue

        idx_prev = int(valid_idx[pos_prev])
        idx_next = int(valid_idx[pos_next])
        gap_prev_min = (fechas_ns_nan_real[start_nan] - fechas_ns_nan_real[idx_prev]) / 1e9 / 60.0
        gap_next_min = (fechas_ns_nan_real[idx_next] - fechas_ns_nan_real[end_nan]) / 1e9 / 60.0
        if gap_prev_min > ANCHOR_MAX_GAP_MIN_NAN_REAL or gap_next_min > ANCHOR_MAX_GAP_MIN_NAN_REAL:
            continue

        delta_anclas = abs(float(valores_nan[idx_next]) - float(valores_nan[idx_prev]))
        if not np.isfinite(delta_anclas) or delta_anclas > delta_max_sensor:
            continue

        recon_real_arr[start_nan:end_nan + 1, j] = True

nan_real_mat = pd.DataFrame(nan_real_arr, index=df_original.index, columns=SENSORES_NAN_REAL)
recon_real_mat = pd.DataFrame(recon_real_arr, index=df_original.index, columns=SENSORES_NAN_REAL)

# Defensa adicional: aunque la deteccion previa ya aplica max_gap, volvemos a
# limitar por duracion de segmento para impedir que cortes largos entren como
# Datos Faltantes corregibles.
_short_gap_arr = np.zeros_like(recon_real_arr, dtype=bool)
for _j, _sensor_nan in enumerate(SENSORES_NAN_REAL):
    _mask_nan_sensor = nan_real_arr[:, _j]
    if not _mask_nan_sensor.any():
        continue
    _padded = np.r_[False, _mask_nan_sensor, False]
    _ch = np.flatnonzero(_padded[1:] != _padded[:-1])
    for _st, _en in zip(_ch[::2], _ch[1::2] - 1):
        _dur_min = ((fechas_ns_nan_real[_en] - fechas_ns_nan_real[_st]) / 1e9 / 60.0) + STEP_MIN_NAN_REAL
        if _dur_min <= MAX_GAP_MIN_NAN_REAL:
            _short_gap_arr[_st:_en + 1, _j] = True
_short_gap_mat = pd.DataFrame(_short_gap_arr, index=df_original.index, columns=SENSORES_NAN_REAL)
recon_real_mat = recon_real_mat & _short_gap_mat
no_corr_real_mat = nan_real_mat & ~recon_real_mat

mask_recon_nan = recon_real_mat.any(axis=1)
mask_no_corr_nan = no_corr_real_mat.any(axis=1)
mask_normal_previa = df_trabajo['etiqueta_deteccion'].eq('normal')

# Si una fila tiene al menos un sensor NaN no corregible, se excluye completa:
# no queremos entrenar una fila parcialmente rota como si fuese un caso limpio.
mask_excluir_nan = mask_no_corr_nan & mask_normal_previa
mask_recon_limpio = mask_recon_nan & ~mask_no_corr_nan & mask_normal_previa

def _combos_sensores_desde_matriz(matriz_bool, mask_filas):
    cols = np.array(matriz_bool.columns, dtype=object)
    sub = matriz_bool.loc[mask_filas].to_numpy(dtype=bool)
    return [','.join(cols[row]) for row in sub]

if mask_recon_limpio.any():
    df_trabajo.loc[mask_recon_limpio, 'etiqueta_deteccion'] = 'anomalia'
    df_trabajo.loc[mask_recon_limpio, 'etiqueta_tipo_anomalia'] = 'Datos Faltantes'
    df_trabajo.loc[mask_recon_limpio, 'sensor_anomalia'] = _combos_sensores_desde_matriz(recon_real_mat, mask_recon_limpio)
    df_trabajo.loc[mask_recon_limpio, 'fuente_anomalia'] = 'nan_real_corto_suave'
    df_trabajo.loc[mask_recon_limpio, COLUMNA_USAR_EN_TRAIN] = True

    # Balance full_v3 por segmentos completos. No muestreamos filas sueltas:
    # eso podria reconstruir fragmentos dentro de cortes largos. Solo entran
    # segmentos completos <= max_gap y hasta un maximo por combinacion.
    _max_real_df_global = int(DATOS_FALTANTES_REAL_MAX_TRAIN_FULL_V3)
    _max_real_df_combo = int(DATOS_FALTANTES_REAL_MAX_POR_COMBO_FULL_V3)
    _idx_keep_parts = []
    _real_combos = df_trabajo.loc[mask_recon_limpio, 'sensor_anomalia'].astype(str)

    def _segmentos_bool_idx_02(_mask):
        _mask = pd.Series(_mask, index=df_trabajo.index).fillna(False).astype(bool)
        if not _mask.any():
            return []
        _gid = _mask.ne(_mask.shift(fill_value=False)).cumsum()
        return [g.index for _, g in _mask[_mask].groupby(_gid[_mask])]

    def _sensor_match_series_02(_series, _sensor):
        _s = _series.fillna('').astype(str)
        return _s.eq(_sensor) | _s.str.split(',').apply(lambda vals: _sensor in [v.strip() for v in vals])

    for _combo in sorted(_real_combos.unique()):
        _mask_combo = mask_recon_limpio & df_trabajo['sensor_anomalia'].astype(str).eq(_combo)
        _segmentos_validos = []
        for _seg_idx in _segmentos_bool_idx_02(_mask_combo):
            _fechas_seg = pd.to_datetime(df_trabajo.loc[_seg_idx, 'Fecha'])
            _dur_min = ((_fechas_seg.iloc[-1] - _fechas_seg.iloc[0]).total_seconds() / 60.0 + STEP_MIN_NAN_REAL) if len(_fechas_seg) > 1 else STEP_MIN_NAN_REAL
            if _dur_min <= MAX_GAP_MIN_NAN_REAL:
                _segmentos_validos.append(_seg_idx)
        if not _segmentos_validos:
            continue
        _seg_meta = []
        for _seg_idx in _segmentos_validos:
            _start_ns = int(pd.to_datetime(df_trabajo.loc[_seg_idx[0], 'Fecha']).value)
            _h = int(pd.util.hash_pandas_object(pd.Series([_start_ns + int(DATOS_FALTANTES_REAL_BALANCE_SEED_FULL_V3)]), index=False).iloc[0])
            _seg_meta.append((_h, _seg_idx))
        _seg_meta.sort(key=lambda x: x[0])
        _combo_keep = []
        _combo_n = 0
        for _, _seg_idx in _seg_meta:
            if _combo_n + len(_seg_idx) > _max_real_df_combo:
                continue
            _combo_keep.append(_seg_idx)
            _combo_n += len(_seg_idx)
        if _combo_keep:
            _idx_keep_parts.append(_combo_keep[0].append(_combo_keep[1:]) if len(_combo_keep) > 1 else _combo_keep[0])

    _idx_keep_real_df = pd.Index([])
    if _idx_keep_parts:
        _idx_keep_real_df = _idx_keep_parts[0].append(_idx_keep_parts[1:])

    # Limite global por seguridad, tambien por segmentos completos.
    if len(_idx_keep_real_df) > _max_real_df_global:
        _mask_keep_tmp = pd.Series(False, index=df_trabajo.index)
        _mask_keep_tmp.loc[_idx_keep_real_df] = True
        _seg_meta_global = []
        for _seg_idx in _segmentos_bool_idx_02(_mask_keep_tmp):
            _start_ns = int(pd.to_datetime(df_trabajo.loc[_seg_idx[0], 'Fecha']).value)
            _h = int(pd.util.hash_pandas_object(pd.Series([_start_ns + int(DATOS_FALTANTES_REAL_BALANCE_SEED_FULL_V3)]), index=False).iloc[0])
            _seg_meta_global.append((_h, _seg_idx))
        _seg_meta_global.sort(key=lambda x: x[0])
        _global_keep = []
        _global_n = 0
        for _, _seg_idx in _seg_meta_global:
            if _global_n + len(_seg_idx) > _max_real_df_global:
                continue
            _global_keep.append(_seg_idx)
            _global_n += len(_seg_idx)
        _idx_keep_real_df = _global_keep[0].append(_global_keep[1:]) if len(_global_keep) > 1 else (_global_keep[0] if _global_keep else pd.Index([]))

    _mask_keep_real_df = pd.Series(False, index=df_trabajo.index)
    if len(_idx_keep_real_df):
        _mask_keep_real_df.loc[_idx_keep_real_df] = True

    # Guardia final por sensor: el balance por combinacion puede dejar segmentos
    # adyacentes que, vistos desde un sensor concreto, superen el max_gap.
    _removed_sensor_guard_segments = 0
    _removed_sensor_guard_rows = 0
    for _sensor_nan_guard in SENSORES_NAN_REAL:
        _mask_selected_sensor = (
            mask_recon_limpio
            & _mask_keep_real_df
            & _sensor_match_series_02(df_trabajo['sensor_anomalia'], _sensor_nan_guard)
        )
        for _seg_idx in _segmentos_bool_idx_02(_mask_selected_sensor):
            _fechas_seg = pd.to_datetime(df_trabajo.loc[_seg_idx, 'Fecha'])
            _dur_min = ((_fechas_seg.iloc[-1] - _fechas_seg.iloc[0]).total_seconds() / 60.0 + STEP_MIN_NAN_REAL) if len(_fechas_seg) > 1 else STEP_MIN_NAN_REAL
            if _dur_min > MAX_GAP_MIN_NAN_REAL:
                _rows_before = int(_mask_keep_real_df.loc[_seg_idx].sum())
                if _rows_before:
                    _mask_keep_real_df.loc[_seg_idx] = False
                    _removed_sensor_guard_segments += 1
                    _removed_sensor_guard_rows += _rows_before
    if _removed_sensor_guard_segments:
        print(
            f"  Balance: guardia por sensor excluye {_removed_sensor_guard_rows:,} filas "
            f"en {_removed_sensor_guard_segments:,} segmentos > {MAX_GAP_MIN_NAN_REAL:.1f} min"
        )

    _mask_excluir_real_df_balance = mask_recon_limpio & ~_mask_keep_real_df
    if _mask_excluir_real_df_balance.any():
        df_trabajo.loc[_mask_excluir_real_df_balance, COLUMNA_USAR_EN_TRAIN] = False
        df_trabajo.loc[_mask_excluir_real_df_balance, 'etiqueta_deteccion'] = 'excluir'
        df_trabajo.loc[_mask_excluir_real_df_balance, 'etiqueta_tipo_anomalia'] = 'NaN Real Corregible Excluido Train'
        df_trabajo.loc[_mask_excluir_real_df_balance, 'fuente_anomalia'] = 'nan_real_corto_suave_no_train'
    mask_recon_limpio_train = mask_recon_limpio & _mask_keep_real_df

if mask_excluir_nan.any():
    df_trabajo.loc[mask_excluir_nan, COLUMNA_USAR_EN_TRAIN] = False
    df_trabajo.loc[mask_excluir_nan, 'etiqueta_deteccion'] = 'excluir'
    df_trabajo.loc[mask_excluir_nan, 'etiqueta_tipo_anomalia'] = 'NaN No Corregible'
    df_trabajo.loc[mask_excluir_nan, 'sensor_anomalia'] = _combos_sensores_desde_matriz(no_corr_real_mat, mask_excluir_nan)
    df_trabajo.loc[mask_excluir_nan, 'fuente_anomalia'] = 'nan_real_no_corregible'

print('Etiquetado NaN real operativo full_v3:')
print(f"  NaNs reales reconstruibles detectados: {int(mask_recon_limpio.sum()):,} filas")
print(f"  NaNs reales reconstruibles usados en train: {int(mask_recon_limpio_train.sum()) if 'mask_recon_limpio_train' in locals() else 0:,} filas")
print(f"  NaNs reales reconstruibles excluidos por balance: {int(_mask_excluir_real_df_balance.sum()) if '_mask_excluir_real_df_balance' in locals() else 0:,} filas")
print(f"  Filas con NaN real no corregible excluidas de entrenamiento: {int(mask_excluir_nan.sum()):,}")
print(f"  Tiempo etiquetado NaN real: {_time_nan_real.time() - _t0_nan_real:.1f} s")
if 'mask_recon_limpio_train' in locals() and mask_recon_limpio_train.any():
    print('  Datos Faltantes reales usados en train por sensor/combinacion:')
    print(df_trabajo.loc[mask_recon_limpio_train, 'sensor_anomalia'].value_counts().sort_index().to_string())
if mask_excluir_nan.any():
    print('  NaN no corregible por combinacion de sensores (top 10):')
    print(df_trabajo.loc[mask_excluir_nan, 'sensor_anomalia'].value_counts().head(10).to_string())


In [ ]:
# Verificacion: a partir de aqui solo deben quedar filas utilizables.
n_no_train = 0
if COLUMNA_USAR_EN_TRAIN in df_trabajo.columns:
    n_no_train = int((~df_trabajo[COLUMNA_USAR_EN_TRAIN].fillna(False).astype(bool)).sum())
print(f"Filas usar_en_train=False presentes tras el filtro: {n_no_train:,}")
print(f"Filas disponibles para inyeccion: {(df_trabajo['etiqueta_deteccion'] == 'normal').sum():,}")


## 1. Columnas objetivo
Definimos las columnas de sensores sobre las que inyectaremos anomalÃ­as (UVENT excluido por seÃ±al no fiable, v3 M3).

In [ ]:
# v6: UVENT_cen y UVENT_lN disponibles como contexto del modelo.
# No se inyectan anomalÃ­as directamente sobre UVENT para no convertir la apertura
# en un objetivo artificial; se reservan para contexto/correlaciÃ³n.
# XTS excluido: 95%+ muestras planas -> SA indistinguible de comportamiento natural del suelo.
columnas_sensores_y_actuadores = [col for col in COLUMNAS_SENSORES if col in df_trabajo.columns]

columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]
print(f"Columnas para inyecciÃ³n de anomalÃ­as: {columnas_objetivo_existentes}")
std_devs_originales = {col: df_original[col].std() for col in columnas_objetivo_existentes if col in df_original.columns}
print(f"Columnas objetivo ({len(columnas_objetivo_existentes)}): {columnas_objetivo_existentes}")


## 2. Sensor Atascado full_v3 (v25 final)

Integra la l?gica final validada en `02b_inyeccion_anomalias_stuck_v25.ipynb`: excluye `PRAD/PRGINT`, incluye `PTEXT/XTINV` con regla especial y usa duraciones compatibles con la correcci?n `v25f`.


In [ ]:
# --- Sensor Atascado full_v3: configuraci?n v25 final ---
RANDOM_STATE = 42
FECHA_TEST_INICIO = INYECCION.get('sensor_atascado', {}).get('fecha_inicio_test_final', '2021-11-05 20:06:30')

SENSORES_STUCK_V25 = ['PCO2EXT', 'PHEXT', 'PTEXT', 'PVV', 'XCO2I', 'XHINV', 'XTINV']
SENSORES_TEMP_ESPECIALES_V25 = {'PTEXT', 'XTINV'}

STUCK_DURACION_V25 = {
    'PCO2EXT': {'duracion_min': 20, 'duracion_max': 40},
    'PHEXT':   {'duracion_min': 16, 'duracion_max': 30},
    'PTEXT':   {'duracion_min': 40, 'duracion_max': 60},
    'PVV':     {'duracion_min': 20, 'duracion_max': 60},
    'XCO2I':   {'duracion_min': 20, 'duracion_max': 60},
    'XHINV':   {'duracion_min': 16, 'duracion_max': 30},
    'XTINV':   {'duracion_min': 40, 'duracion_max': 60},
}

STUCK_MIN_SIGMA_SEPARACION_V25 = {
    'PCO2EXT': 0.75,
    'PHEXT':   0.50,
    'PTEXT':   0.50,
    'PVV':     0.75,
    'XCO2I':   1.00,
    'XHINV':   0.50,
    'XTINV':   0.50,
}

TEMP_PERIODOS_PERMITIDOS_V25 = {'morning', 'afternoon'}
TEMP_PRAD_MEDIA_MIN_V25 = 5.0
TEMP_DELTA_MIN_ABS_V25 = {'PTEXT': 1.50, 'XTINV': 1.50}
TEMP_DELTA_MAX_ABS_V25 = {'PTEXT': 4.0, 'XTINV': 5.0}
TEMP_RANGO_MIN_ABS_V25 = {'PTEXT': 0.75, 'XTINV': 0.75}

XCO2I_PREV_OUTLIER_MAX_SIGMA_V25 = 1.0
XCO2I_SEV_MAX_SIGMA_V25 = 5.0
XCO2I_FRAC_GT3_MAX_V25 = 0.10
XCO2I_FRAC_DEBIL_MAX_V25 = 0.60
XCO2I_PREV_WINDOW_V25 = 10

CUOTAS_STUCK_V25 = {
    'train': {'PCO2EXT': 40, 'PHEXT': 40, 'PTEXT': 40, 'PVV': 40, 'XCO2I': 40, 'XHINV': 40, 'XTINV': 40},
    'test':  {'PCO2EXT': 10, 'PHEXT': 10, 'PTEXT': 10, 'PVV': 10, 'XCO2I': 10, 'XHINV': 10, 'XTINV': 10},
}

SENSORES_STUCK_V25 = [
    s for s in SENSORES_STUCK_V25
    if s in df_trabajo.columns and s not in STUCK_SA_EXCLUIR_CORRECCION
]
std_devs_stuck_v25 = {
    col: float(pd.to_numeric(df_original[col], errors='coerce').std())
    for col in SENSORES_STUCK_V25
}
print('Sensores SA full_v3:', SENSORES_STUCK_V25)
print('Inicio test SA:', FECHA_TEST_INICIO)

def estacion_meteorologica(mes):
    if mes in (12, 1, 2):
        return 'winter'
    if mes in (3, 4, 5):
        return 'spring'
    if mes in (6, 7, 8):
        return 'summer'
    return 'autumn'


def repartir_cuota_por_estacion(candidatos_por_estacion, objetivo):
    total = sum(len(v) for v in candidatos_por_estacion.values())
    if objetivo <= 0 or total == 0:
        return {k: 0 for k in candidatos_por_estacion}

    bruto = {k: objetivo * len(v) / total for k, v in candidatos_por_estacion.items()}
    cuotas = {k: min(len(candidatos_por_estacion[k]), int(np.floor(v))) for k, v in bruto.items()}
    pendientes = objetivo - sum(cuotas.values())

    orden = sorted(
        candidatos_por_estacion,
        key=lambda k: (bruto[k] - np.floor(bruto[k]), len(candidatos_por_estacion[k])),
        reverse=True,
    )
    for k in orden:
        if pendientes <= 0:
            break
        if cuotas[k] < len(candidatos_por_estacion[k]):
            cuotas[k] += 1
            pendientes -= 1
    return cuotas


def periodo_dia(hora):
    if 6 <= hora < 12:
        return 'morning'
    if 12 <= hora < 18:
        return 'afternoon'
    if 18 <= hora < 22:
        return 'evening'
    return 'night'


def candidatos_stuck_v25(df, sensor, idx_inicio, idx_fin):
    serie = pd.to_numeric(df[sensor], errors='coerce')
    valores = serie.to_numpy(dtype=float)
    previos = np.roll(valores, 1)
    previos[0] = np.nan

    cfg = STUCK_DURACION_V25.get(sensor, STUCK_DURACION_POR_SENSOR[sensor])
    dur_min = int(cfg['duracion_min'])
    dur_max = int(cfg['duracion_max'])
    sigma = std_devs_stuck_v25.get(sensor, 0.0)
    min_sigma = STUCK_MIN_SIGMA_SEPARACION_V25.get(sensor, STUCK_MIN_SIGMA_SEPARACION)
    umbral_sep = max(min_sigma * sigma if sigma > 0 else 0.0, 1e-6)

    idx_inicio = max(0, int(idx_inicio))
    idx_fin = min(len(df), int(idx_fin))
    max_start = idx_fin - dur_min
    if max_start < idx_inicio:
        return pd.DataFrame(columns=['start', 'dur_min', 'dur_max', 'season', 'period'])

    if sensor in SENSORES_TEMP_ESPECIALES_V25:
        valido = (
            df['etiqueta_deteccion'].eq('normal').to_numpy() &
            np.isfinite(valores) &
            np.isfinite(previos) &
            ~np.isclose(valores, previos, rtol=0.0, atol=1e-12)
        )
    else:
        # Regla general v2.1: ventana local con variabilidad suficiente y separacion minima.
        rstd = serie.rolling(30, min_periods=10).std().to_numpy(dtype=float)
        positivos = rstd[np.isfinite(rstd) & (rstd > 0)]
        umbral_var = float(np.percentile(positivos, 20)) if len(positivos) else 0.0
        valido = (
            df['etiqueta_deteccion'].eq('normal').to_numpy() &
            np.isfinite(valores) &
            np.isfinite(previos) &
            (rstd >= umbral_var) &
            ~np.isclose(valores, previos, rtol=0.0, atol=1e-12)
        )

    vmin = STUCK_VALOR_MINIMO_INYECCION.get(sensor)
    if vmin is not None:
        valido &= (valores >= vmin) & (previos >= vmin)

    if sensor in RANGOS_FISICOS:
        r = RANGOS_FISICOS[sensor]
        valido &= (
            (valores >= r['min']) & (valores <= r['max']) &
            (previos >= r['min']) & (previos <= r['max'])
        )

    cum = np.concatenate([[0], np.cumsum(valido.astype(np.int32))])
    starts = np.arange(idx_inicio, max_start + 1)
    sumas = cum[starts + dur_min] - cum[starts]
    starts = starts[sumas == dur_min]

    buenos = []
    for st in starts:
        stuck_value = previos[st]
        ventana = valores[st:st + dur_min]
        if len(ventana) < dur_min or not np.isfinite(stuck_value) or not np.isfinite(ventana).all():
            continue

        if sensor in SENSORES_TEMP_ESPECIALES_V25:
            fecha_st = df.loc[st, 'Fecha']
            period = periodo_dia(fecha_st.hour)
            if period not in TEMP_PERIODOS_PERMITIDOS_V25:
                continue
            if 'PRAD' in df.columns:
                prad = pd.to_numeric(df.loc[st:st + dur_min - 1, 'PRAD'], errors='coerce').to_numpy(dtype=float)
                if len(prad) < dur_min or not np.isfinite(prad).all() or float(np.mean(prad)) < TEMP_PRAD_MEDIA_MIN_V25:
                    continue

            diffs = np.abs(ventana - stuck_value)
            diff_mean = float(np.mean(diffs))
            diff_max = float(np.max(diffs))
            rango = float(np.max(ventana) - np.min(ventana))
            delta_min = TEMP_DELTA_MIN_ABS_V25[sensor]
            delta_max = TEMP_DELTA_MAX_ABS_V25[sensor]
            rango_min = TEMP_RANGO_MIN_ABS_V25[sensor]
            if diff_mean < delta_min and diff_max < 1.5 * delta_min:
                continue
            if diff_max > delta_max:
                continue
            if rango < rango_min and abs(float(ventana[-1]) - float(stuck_value)) < delta_min:
                continue
        else:
            diff_med = float(np.median(np.abs(ventana - stuck_value)))
            diff_mean = float(abs(np.mean(ventana) - stuck_value))
            diff_max = float(np.max(np.abs(ventana - stuck_value)))
            if diff_med < umbral_sep and diff_mean < umbral_sep and diff_max < 2.0 * umbral_sep:
                continue
        buenos.append(st)

    if not buenos:
        return pd.DataFrame(columns=['start', 'dur_min', 'dur_max', 'season', 'period'])

    out = pd.DataFrame({'start': buenos})
    out['dur_min'] = dur_min
    out['dur_max'] = dur_max
    fechas_start = df.loc[out['start'], 'Fecha']
    out['season'] = fechas_start.dt.month.map(estacion_meteorologica).to_numpy()
    out['period'] = fechas_start.dt.hour.map(periodo_dia).to_numpy()
    return out


In [ ]:
fecha_corte = pd.Timestamp(FECHA_TEST_INICIO)
fechas = pd.to_datetime(df_trabajo['Fecha'])
idx_test_inicio_arr = np.flatnonzero(fechas >= fecha_corte)
idx_test_inicio = int(idx_test_inicio_arr[0]) if len(idx_test_inicio_arr) else len(df_trabajo)

candidatos_cache = {}
resumen_candidatos = []

for split, idx_a, idx_b in [('train', 0, idx_test_inicio), ('test', idx_test_inicio, len(df_trabajo))]:
    for sensor in SENSORES_STUCK_V25:
        cand = candidatos_stuck_v25(df_trabajo, sensor, idx_a, idx_b)
        candidatos_cache[(split, sensor)] = cand
        por_estacion = cand['season'].value_counts().to_dict() if not cand.empty else {}
        resumen_candidatos.append({
            'split': split,
            'sensor': sensor,
            'candidatos': len(cand),
            **{f'cand_{k}': por_estacion.get(k, 0) for k in ['winter', 'spring', 'summer', 'autumn']},
        })

df_resumen_candidatos = pd.DataFrame(resumen_candidatos)
print(df_resumen_candidatos.to_string(index=False))


def stable_seed(base_seed, *parts):
    texto = '|'.join([str(base_seed), *map(str, parts)])
    digest = hashlib.blake2b(texto.encode('utf-8'), digest_size=8).digest()
    return int.from_bytes(digest, 'little') % (2**32)


def validar_xco2i_evento_v25(df_ref, sensor, st, end, prev_value):
    sigma = std_devs_originales.get(sensor, np.nan)
    if sensor != 'XCO2I' or not np.isfinite(sigma) or sigma <= 0:
        return True

    valores = pd.to_numeric(df_ref.iloc[st:end][sensor], errors='coerce').to_numpy(dtype=float)
    if len(valores) == 0 or not np.isfinite(valores).all() or not np.isfinite(prev_value):
        return False

    prev_start = max(0, st - XCO2I_PREV_WINDOW_V25)
    prev_window = pd.to_numeric(df_ref.iloc[prev_start:st][sensor], errors='coerce').dropna().to_numpy(dtype=float)
    if len(prev_window) >= 3:
        prev_med = float(np.median(prev_window))
        if abs(float(prev_value) - prev_med) / sigma > XCO2I_PREV_OUTLIER_MAX_SIGMA_V25:
            return False

    sev = np.abs(valores - float(prev_value)) / sigma
    if float(np.max(sev)) > XCO2I_SEV_MAX_SIGMA_V25:
        return False
    if float(np.mean(sev > 3.0)) > XCO2I_FRAC_GT3_MAX_V25:
        return False
    if float(np.mean(sev < 0.5)) > XCO2I_FRAC_DEBIL_MAX_V25:
        return False
    return True


def inyectar_stuck_v25(df, candidatos_cache, cuotas, random_state=42):
    df_mod = df.copy()
    usados = np.zeros(len(df_mod), dtype=bool)
    registros = []
    anomalia_counter = 1

    col_pos = {s: df_mod.columns.get_loc(s) for s in SENSORES_STUCK_V25}
    det_idx = df_mod.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_mod.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_mod.columns.get_loc('sensor_anomalia')
    id_idx = df_mod.columns.get_loc('anomalia_id')
    season_idx = df_mod.columns.get_loc('season_anomalia')
    period_idx = df_mod.columns.get_loc('period_anomalia')

    for split, cuotas_sensor in cuotas.items():
        for sensor, objetivo in cuotas_sensor.items():
            if sensor not in SENSORES_STUCK_V25:
                continue
            cand = candidatos_cache.get((split, sensor), pd.DataFrame()).copy()
            if cand.empty:
                print(f'Aviso: {sensor} [{split}] sin candidatos.')
                continue

            cand_por_estacion = {
                season: cand.loc[cand['season'].eq(season), 'start'].to_numpy(dtype=int)
                for season in ['winter', 'spring', 'summer', 'autumn']
            }
            cuotas_estacion = repartir_cuota_por_estacion(cand_por_estacion, int(objetivo))
            creadas = 0
            rechazadas_xco2i = 0

            for season, n_season in cuotas_estacion.items():
                if n_season <= 0:
                    continue

                rng = np.random.default_rng(stable_seed(random_state, 'stuck_v23', split, sensor, season))
                starts = rng.permutation(cand_por_estacion[season])

                for st in starts:
                    if creadas >= objetivo or n_season <= 0:
                        break
                    dur_min = int(cand.loc[cand['start'].eq(st), 'dur_min'].iloc[0])
                    dur_max = int(cand.loc[cand['start'].eq(st), 'dur_max'].iloc[0])
                    dur = int(rng.integers(dur_min, dur_max + 1))
                    end = st + dur
                    if end > len(df_mod):
                        continue
                    if usados[st:end].any():
                        continue
                    if not df_mod.loc[st:end - 1, 'etiqueta_deteccion'].eq('normal').all():
                        continue
                    valores = pd.to_numeric(df_mod.iloc[st:end, col_pos[sensor]], errors='coerce').to_numpy(dtype=float)
                    prev_value = pd.to_numeric(df_mod.iloc[st - 1, col_pos[sensor]], errors='coerce') if st > 0 else np.nan
                    if not np.isfinite(prev_value) or not np.isfinite(valores).all():
                        continue
                    if sensor == 'XCO2I' and not validar_xco2i_evento_v25(df_mod, sensor, st, end, float(prev_value)):
                        rechazadas_xco2i += 1
                        continue

                    df_mod.iloc[st:end, col_pos[sensor]] = float(prev_value)
                    df_mod.iloc[st:end, det_idx] = 'anomalia'
                    df_mod.iloc[st:end, tipo_idx] = 'Sensor Atascado'
                    df_mod.iloc[st:end, sensor_idx] = sensor
                    df_mod.iloc[st:end, id_idx] = f'SA25_{anomalia_counter:06d}'
                    period = cand.loc[cand['start'].eq(st), 'period'].iloc[0] if 'period' in cand.columns else pd.NA
                    df_mod.iloc[st:end, season_idx] = season
                    df_mod.iloc[st:end, period_idx] = period
                    usados[st:end] = True

                    registros.append({
                        'anomalia_id': f'SA25_{anomalia_counter:06d}',
                        'split': split,
                        'sensor': sensor,
                        'season': season,
                        'period': period,
                        'start_idx': st,
                        'end_idx': end - 1,
                        'duracion': dur,
                        'stuck_value': float(prev_value),
                        'rng_seed': stable_seed(random_state, 'stuck_v23', split, sensor, season),
                    })
                    anomalia_counter += 1
                    creadas += 1
                    n_season -= 1

            estado = 'OK' if creadas >= objetivo else f'AVISO: {creadas}/{objetivo}'
            extra = f' | XCO2I rechazadas: {rechazadas_xco2i}' if sensor == 'XCO2I' else ''
            print(f'{sensor:8s} [{split:5s}]: {creadas:3d} secuencias {estado}{extra}')

    return df_mod, pd.DataFrame(registros)



# --- Aplicar Sensor Atascado full_v3 ---
etiquetas_deteccion_antes_stuck = df_trabajo['etiqueta_deteccion'].value_counts(dropna=False)
etiquetas_tipo_antes_stuck = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

df_trabajo, df_eventos_stuck_full_v3 = inyectar_stuck_v25(df_trabajo, candidatos_cache, CUOTAS_STUCK_V25, RANDOM_STATE)
print('\nEventos SA full_v3 creados:', len(df_eventos_stuck_full_v3))
if not df_eventos_stuck_full_v3.empty:
    print(df_eventos_stuck_full_v3.groupby(['split', 'sensor', 'season']).size().unstack(fill_value=0).to_string())
    print(df_eventos_stuck_full_v3.groupby(['split', 'sensor', 'period']).size().unstack(fill_value=0).to_string())
    df_eventos_stuck_full_v3.to_csv(OUT_EVENTOS_STUCK_FULL_V3, index=False)
    print(f'Guardado eventos SA full_v3: {OUT_EVENTOS_STUCK_FULL_V3}')

print('--- Resultados de la Inyeccion de Sensor Atascado full_v3 ---')
print('Conteo etiqueta_deteccion ANTES:')
print(etiquetas_deteccion_antes_stuck)
print('Conteo etiqueta_deteccion DESPUES:')
print(df_trabajo['etiqueta_deteccion'].value_counts(dropna=False))
print('Conteo etiqueta_tipo_anomalia ANTES:')
print(etiquetas_tipo_antes_stuck)
print('Conteo etiqueta_tipo_anomalia DESPUES:')
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
print('\nSensor Atascado por sensor_anomalia:')
print(df_trabajo.loc[df_trabajo['etiqueta_tipo_anomalia'].eq('Sensor Atascado'), 'sensor_anomalia'].value_counts(dropna=False).sort_index())

# Auditoria rapida SA full_v3: constantes, NaN y solapes dentro de SA.
_rows_audit_sa = []
_mask_sa = df_trabajo['etiqueta_tipo_anomalia'].eq('Sensor Atascado') & df_trabajo['sensor_anomalia'].notna()
for (_sensor, _aid), _g in df_trabajo.loc[_mask_sa].groupby(['sensor_anomalia', 'anomalia_id']):
    _rows_audit_sa.append({
        'sensor': _sensor,
        'anomalia_id': _aid,
        'n': len(_g),
        'nunique_inyectado': int(pd.to_numeric(_g[_sensor], errors='coerce').nunique(dropna=True)),
        'nan_en_evento': int(_g[_sensor].isna().sum()),
    })
df_audit_stuck_full_v3 = pd.DataFrame(_rows_audit_sa)
if not df_audit_stuck_full_v3.empty:
    df_audit_stuck_full_v3.to_csv(OUT_AUDITORIA_STUCK_FULL_V3, index=False)
    print('\nAuditoria SA full_v3:')
    print(df_audit_stuck_full_v3.groupby('sensor').agg(eventos=('anomalia_id','count'), min_n=('n','min'), median_n=('n','median'), max_n=('n','max'), non_const=('nunique_inyectado', lambda x: int((x != 1).sum())), with_nan=('nan_en_evento', lambda x: int((x > 0).sum()))).to_string())
else:
    print('Auditoria SA full_v3: no hay eventos SA.')


## 3. Datos Faltantes
 
 En este experimento los `NaN` originales del dataset se tratan como faltantes reales, no como anomalÃ­as sintÃ©ticas. Para mantener un ground truth evaluable, la inyecciÃ³n de `Datos Faltantes` solo se aplica sobre filas normales completas, donde el valor limpio existe en `PARQUET_01` y puede compararse despuÃ©s con la reconstrucciÃ³n.
 
 Si una fila ya contiene `NaN` en algÃºn sensor objetivo antes de la inyecciÃ³n, se excluye de los candidatos y no se etiqueta como `Datos Faltantes`. AsÃ­ evitamos medir RMSE/MAE contra un valor verdadero desconocido.


In [ ]:
def inyectar_datos_faltantes(df, porcentaje_filas_afectadas, columnas_objetivo,
                             num_valores_nan_por_fila=1, random_state=42):
    """Inyecta NaN solo en filas normales completas, sin etiquetar NaN reales del dataset."""
    if not columnas_objetivo:
        print('No se proporcionaron columnas objetivo.')
        return df

    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    cols = [c for c in columnas_objetivo if c in df_modificado.columns]
    if not cols:
        print('Ninguna columna objetivo existe en el dataframe.')
        return df_modificado

    # --- Mascara vectorizada ---
    mask_normal = (df_modificado['etiqueta_deteccion'] == 'normal').to_numpy()
    col_arr = df_modificado[cols].to_numpy(dtype=float)  # (N, n_cols)

    # Columnas validas por fila: no NaN y por encima del valor minimo de inyeccion
    col_valid = ~np.isnan(col_arr)
    for j, col in enumerate(cols):
        vmin = DATOS_FALTANTES_VALOR_MINIMO_INYECCION.get(col)
        if vmin is not None:
            col_valid[:, j] &= col_arr[:, j] >= vmin

    # Candidatas: filas normales con todas las columnas sin NaN (conserva semantica original)
    mask_completa = ~np.isnan(col_arr).any(axis=1)
    mask_candidatos = mask_normal & mask_completa
    idx_candidatos = np.flatnonzero(mask_candidatos)

    filas_excluidas = int(mask_normal.sum()) - len(idx_candidatos)
    if filas_excluidas:
        print(f'Filas normales excluidas por NaNs reales en sensores objetivo: {filas_excluidas}')

    n_objetivo = int(len(idx_candidatos) * porcentaje_filas_afectadas)
    if n_objetivo == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado

    n_sample = min(n_objetivo, len(idx_candidatos))
    print(f'Inyectando datos faltantes en hasta {n_sample} filas...')

    # Seleccion vectorizada de filas
    sel_local = rng.choice(len(idx_candidatos), size=n_sample, replace=False)
    filas_sel = idx_candidatos[sel_local]

    # Seleccion vectorizada de columna por fila (truco noise: argmax sobre columnas validas)
    valid_sel = col_valid[filas_sel]
    noise = rng.uniform(0, 1, valid_sel.shape)
    noise[~valid_sel] = -1.0
    col_chosen = np.argmax(noise, axis=1)  # indice local dentro de cols

    # Inyeccion de NaN por columna en bulk
    col_pos = [df_modificado.columns.get_loc(c) for c in cols]
    for j in range(len(cols)):
        rows_j = filas_sel[col_chosen == j]
        if len(rows_j):
            df_modificado.iloc[rows_j, col_pos[j]] = np.nan

    # Etiquetas en bulk
    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None
    df_modificado.iloc[filas_sel, det_idx]  = 'anomalia'
    df_modificado.iloc[filas_sel, tipo_idx] = 'Datos Faltantes'
    if sensor_idx is not None:
        sensores_sel = np.array(cols, dtype=object)[col_chosen]
        df_modificado.iloc[filas_sel, sensor_idx] = sensores_sel

    print(f'Inyeccion de datos faltantes completada. {n_sample} filas modificadas '
          f'y {n_sample} valores convertidos a NaN.')
    return df_modificado


In [ ]:
if 'df_trabajo' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    cfg_faltantes = INYECCION['datos_faltantes']
    porcentaje_filas_nan = cfg_faltantes['porcentaje_filas']
    num_sensores_por_fila_afectada = cfg_faltantes['num_sensores_por_fila']

    etiquetas_deteccion_antes = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    df_trabajo = inyectar_datos_faltantes(
        df_trabajo,
        porcentaje_filas_nan,
        columnas_objetivo_existentes,
        num_sensores_por_fila_afectada,
    )

    print('\n--- Resultados de la Inyeccion de Datos Faltantes ---')
    print('\nConteo etiqueta_deteccion ANTES:')
    print(etiquetas_deteccion_antes)
    print('\nConteo etiqueta_deteccion DESPUES:')
    print(df_trabajo['etiqueta_deteccion'].value_counts())
    print('\nConteo etiqueta_tipo_anomalia ANTES:')
    print(etiquetas_tipo_antes)
    print('\nConteo etiqueta_tipo_anomalia DESPUES:')
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print("Error: df_trabajo o columnas_objetivo_existentes no estan definidos.")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))

In [ ]:
if 'df_trabajo' in locals():
    # Asegurar que el DataFrame estÃ© ordenado por fecha
    if 'Fecha' in df_trabajo.columns:
        print("Asegurando que el DataFrame estÃ© ordenado por 'Fecha'...")
        df_trabajo = df_trabajo.sort_values(by='Fecha').reset_index(drop=True)
        print("DataFrame ordenado por 'Fecha' y el Ã­ndice ha sido reseteado.")
    else:
        print("Advertencia: La columna 'Fecha' no se encontrÃ³. No se puede asegurar el orden temporal para 'Sensor Atascado'.")

else:
    print("Error: 'df_trabajo' no estÃ¡ definido.")

## 4. Ruido / Picos Aleatorios v2
Inyectamos picos de una muestra con severidad espec?fica por sensor y restricciones f?sicas/temporales. Evitamos PRAD nocturno y no usamos sigma global para definir saltos en variables con ciclos fuertes.

In [ ]:
# Reutilizamos la lista definida anteriormente
if 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print(f"Se considerarÃ¡n las siguientes columnas para simular 'Ruido / Picos': \n{columnas_objetivo_existentes}")
else:
    # Esta definiciÃ³n es por si se ejecuta esta celda de forma aislada.
    columnas_sensores_y_actuadores = [col for col in COLUMNAS_SENSORES if col in df_trabajo.columns]
    columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]
    if not columnas_objetivo_existentes:
        print("ADVERTENCIA: 'columnas_objetivo_existentes' estÃ¡ vacÃ­a o no definida.")
    else:
        print(f"Se utilizarÃ¡n las siguientes columnas para simular 'Ruido / Picos': \n{columnas_objetivo_existentes}")

# Calcular la desviaciÃ³n estÃ¡ndar de las columnas objetivo a partir del DataFrame ORIGINAL para tener una medida de variabilidad 'limpia'.
std_devs_originales = {}
if 'df_original' in locals() and columnas_objetivo_existentes:
    print("\nCalculando desviaciones estÃ¡ndar de las columnas objetivo (desde df_original):")
    for col in columnas_objetivo_existentes:
        if pd.api.types.is_numeric_dtype(df_original[col]):
            std_devs_originales[col] = df_original[col].std()
        else:
            print(f"  Advertencia: Columna '{col}' no es numÃ©rica en df_original. No se calcularÃ¡ std.")
            std_devs_originales[col] = 0
    if not std_devs_originales:
        print("Advertencia: No se pudieron calcular desviaciones estÃ¡ndar. Verifica 'df_original'.")
elif not columnas_objetivo_existentes:
    print("Advertencia: 'columnas_objetivo_existentes' no estÃ¡ definida. No se pueden calcular std.")
else:
    print("Advertencia: 'df_original' no estÃ¡ definido. No se pueden calcular std de referencia.")


In [ ]:
# --- Ruido v2: picos realistas por sensor y split -------------------------
# Objetivo: mantener picos claros pero no absurdos fisicamente.
# - PRAD solo diurno y con radiacion real.
# - PRGINT/PVV solo positivos y con valor base minimo. PVV usa spikes > p99 natural de rafagas.
# - Temperatura/humedad usan saltos absolutos razonables, no sigma global.
RUIDO_V2_DELTAS = {
    'PTEXT':   {'min': 3.0,  'max': 8.0,   'signos': [-1, 1]},
    'XTINV':   {'min': 3.0,  'max': 8.0,   'signos': [-1, 1]},
    'PHEXT':   {'min': 10.0, 'max': 25.0,  'signos': [-1, 1]},
    'XHINV':   {'min': 10.0, 'max': 25.0,  'signos': [-1, 1]},
    'PCO2EXT': {'min': 25.0, 'max': 75.0,  'signos': [-1, 1]},
    'XCO2I':   {'min': 60.0, 'max': 160.0, 'signos': [-1, 1]},
    'PVV':     {'min': 5.0,  'max': 9.0,   'signos': [1]},
    'PRAD':    {'min': 150.0,'max': 500.0, 'signos': [1]},
    'PRGINT':  {'min': 80.0, 'max': 250.0, 'signos': [1]},
}

RUIDO_V2_CUOTAS = {
    'train': {s: 5000 for s in RUIDO_V2_DELTAS},
    'test':  {s: 1200 for s in RUIDO_V2_DELTAS},
}
RUIDO_V2_PRAD_MIN = 50.0
RUIDO_V2_PRAD_HORAS = set(range(8, 20))
RUIDO_V2_PRGINT_MIN = 100.0
RUIDO_V2_PVV_MIN = 5.0


def _clip_rango_sensor(col, valor):
    if col not in RANGOS_FISICOS:
        return float(valor)
    r = RANGOS_FISICOS[col]
    span = float(r['max'] - r['min'])
    eps = max(span * 1e-4, 1e-6)
    return float(np.clip(valor, r['min'] + eps, r['max'] - eps))


def _candidatos_ruido_v2(df, sensor, idx_a, idx_b):
    valores = pd.to_numeric(df[sensor], errors='coerce')
    fechas = pd.to_datetime(df['Fecha'], errors='coerce')
    mask = (
        df['etiqueta_deteccion'].eq('normal') &
        valores.notna() &
        fechas.notna()
    )
    if sensor in RANGOS_FISICOS:
        r = RANGOS_FISICOS[sensor]
        mask &= valores.between(r['min'], r['max'])
    if sensor == 'PRAD':
        mask &= (valores >= RUIDO_V2_PRAD_MIN) & fechas.dt.hour.isin(RUIDO_V2_PRAD_HORAS)
    elif sensor == 'PRGINT':
        mask &= valores >= RUIDO_V2_PRGINT_MIN
    elif sensor == 'PVV':
        mask &= valores >= RUIDO_V2_PVV_MIN
    elif sensor == 'PCO2EXT':
        mask &= valores >= RUIDO_VALOR_MINIMO_INYECCION.get('PCO2EXT', -np.inf)

    idx = df.index[mask].to_numpy(dtype=int)
    idx = idx[(idx >= int(idx_a)) & (idx < int(idx_b))]
    return idx


def _generar_valor_ruido_v2(col, val, rng):
    cfg = RUIDO_V2_DELTAS[col]
    for _ in range(20):
        delta = float(rng.uniform(cfg['min'], cfg['max']))
        signo = float(rng.choice(cfg['signos']))
        nuevo = _clip_rango_sensor(col, float(val) + signo * delta)
        cambio = abs(nuevo - float(val))
        if cambio >= cfg['min'] * 0.8:
            return nuevo, cambio, signo
    return np.nan, 0.0, np.nan


def inyectar_ruido_picos(df, porcentaje_filas_ruido, columnas_objetivo, std_devs_ref,
                         factor_ruido_std_min=3, factor_ruido_std_max=5,
                         num_sensores_ruido_por_fila=1, random_state=42):
    """Inyecta ruido v2 con picos de una muestra, realistas por sensor y con cuotas temporales."""
    df_modificado = df.copy()
    sensores = [c for c in columnas_objetivo if c in RUIDO_V2_DELTAS and c in df_modificado.columns]
    if not sensores:
        print('No hay sensores validos para Ruido v2.')
        return df_modificado

    fecha_corte = pd.Timestamp(FECHA_TEST_INICIO)
    fechas = pd.to_datetime(df_modificado['Fecha'], errors='coerce')
    idx_test_inicio_arr = np.flatnonzero(fechas >= fecha_corte)
    idx_test_inicio = int(idx_test_inicio_arr[0]) if len(idx_test_inicio_arr) else len(df_modificado)

    det_idx = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None
    col_indices = {c: df_modificado.columns.get_loc(c) for c in sensores}

    usados = np.zeros(len(df_modificado), dtype=bool)
    registros = []
    resumen = []

    for split, idx_a, idx_b in [('train', 0, idx_test_inicio), ('test', idx_test_inicio, len(df_modificado))]:
        for sensor in sensores:
            objetivo = int(RUIDO_V2_CUOTAS.get(split, {}).get(sensor, 0))
            if objetivo <= 0:
                continue
            cand = _candidatos_ruido_v2(df_modificado, sensor, idx_a, idx_b)
            rng = np.random.default_rng(stable_seed(random_state, 'ruido_v2', split, sensor))
            cand = rng.permutation(cand)
            creadas = 0
            rechazadas = 0

            for fila_idx in cand:
                if creadas >= objetivo:
                    break
                if usados[fila_idx]:
                    continue
                if df_modificado.iat[fila_idx, det_idx] != 'normal':
                    continue
                val = df_modificado.iat[fila_idx, col_indices[sensor]]
                if pd.isna(val):
                    continue
                nuevo, cambio, signo = _generar_valor_ruido_v2(sensor, float(val), rng)
                if not np.isfinite(nuevo) or np.isclose(nuevo, float(val), rtol=0.0, atol=1e-12):
                    rechazadas += 1
                    continue

                df_modificado.iat[fila_idx, col_indices[sensor]] = float(nuevo)
                df_modificado.iat[fila_idx, det_idx] = 'anomalia'
                df_modificado.iat[fila_idx, tipo_idx] = 'Ruido'
                if sensor_idx is not None:
                    df_modificado.iat[fila_idx, sensor_idx] = sensor
                usados[fila_idx] = True
                creadas += 1
                registros.append({
                    'split': split,
                    'sensor': sensor,
                    'idx': int(fila_idx),
                    'Fecha': df_modificado.at[fila_idx, 'Fecha'],
                    'valor_original': float(val),
                    'valor_inyectado': float(nuevo),
                    'delta': float(nuevo - float(val)),
                    'abs_delta': float(cambio),
                    'signo': float(signo),
                    'rng_seed': int(stable_seed(random_state, 'ruido_v2', split, sensor)),
                })

            resumen.append({
                'split': split,
                'sensor': sensor,
                'objetivo': objetivo,
                'candidatos': int(len(cand)),
                'creadas': int(creadas),
                'rechazadas': int(rechazadas),
                'estado': 'OK' if creadas == objetivo else f'AVISO {creadas}/{objetivo}',
            })
            print(f"Ruido v2 {sensor:8s} [{split:5s}]: {creadas:5d}/{objetivo:5d}  candidatos={len(cand):7d}")

    df_resumen_ruido_v2 = pd.DataFrame(resumen)
    df_eventos_ruido_v2 = pd.DataFrame(registros)
    globals()['df_resumen_ruido_v2'] = df_resumen_ruido_v2
    globals()['df_eventos_ruido_v2'] = df_eventos_ruido_v2

    if len(df_eventos_ruido_v2):
        out_ruido = os.path.join(DATA_RESULTS, '02_ruido_full_v3_eventos.csv')
        df_eventos_ruido_v2.to_csv(out_ruido, index=False)
        out_resumen = os.path.join(DATA_RESULTS, '02_ruido_full_v3_resumen.csv')
        df_resumen_ruido_v2.to_csv(out_resumen, index=False)
        print(f'Eventos Ruido v2 guardados: {out_ruido}')
        print(f'Resumen Ruido v2 guardado: {out_resumen}')
        print('\nRuido v2 por sensor/split:')
        print(pd.crosstab(df_eventos_ruido_v2['sensor'], df_eventos_ruido_v2['split']).to_string())
        print('\nDelta abs Ruido v2 por sensor:')
        print(df_eventos_ruido_v2.groupby('sensor')['abs_delta'].agg(['count','min','median','max']).round(3).to_string())

    print('Inyeccion de ruido v2 completada. ' + str(len(registros)) + ' filas modificadas.')
    return df_modificado


In [ ]:
if ('df_trabajo' in locals() and 
    'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes and
    'std_devs_originales' in locals() and std_devs_originales):

    # ParÃ¡metros para la inyecciÃ³n de "Ruido / Picos Aleatorios"
    cfg_ruido = INYECCION['ruido']
    porcentaje_filas_con_ruido = cfg_ruido['porcentaje_filas']
    factor_std_min = cfg_ruido['factor_std_min']
    factor_std_max = cfg_ruido['factor_std_max']
    num_sensores_afectados_por_fila = cfg_ruido['num_sensores_por_fila']

    # Guardar el estado de las etiquetas antes de la inyecciÃ³n para comparar
    etiquetas_deteccion_antes_ruido = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_ruido = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyecciÃ³n de 'Ruido / Picos Aleatorios'...")
    df_trabajo = inyectar_ruido_picos(df_trabajo,
                                        porcentaje_filas_con_ruido,
                                        columnas_objetivo_existentes,
                                        std_devs_originales,
                                        factor_std_min,
                                        factor_std_max,
                                        num_sensores_afectados_por_fila)

    print("\n--- Resultados de la InyecciÃ³n de 'Ruido / Picos Aleatorios' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_ruido)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÃ‰S:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_ruido)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÃ‰S:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    num_ruido_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Ruido').sum()
    print(f"\nNÃºmero total de filas etiquetadas como 'Ruido': {num_ruido_etiquetados}")

else:
    print("Error: AsegÃºrate de que 'df_trabajo', 'columnas_objetivo_existentes' y 'std_devs_originales' estÃ©n definidos.")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())

In [ ]:
# --- Auditoria Ruido full_v3 -------------------------------------------------
if 'df_trabajo' in locals() and 'df_original' in locals():
    mask_ruido_v2 = df_trabajo['etiqueta_tipo_anomalia'].eq('Ruido')
    sensores_audit_ruido = [c for c in columnas_objetivo_existentes if c in df_trabajo.columns and c in df_original.columns]
    filas = []
    cambios_fuera_total = 0
    value_to_nan_total = 0

    for sensor in sensores_audit_ruido:
        mask_sensor = mask_ruido_v2 & df_trabajo['sensor_anomalia'].eq(sensor)
        orig_s = pd.to_numeric(df_original[sensor], errors='coerce')
        inj_s = pd.to_numeric(df_trabajo[sensor], errors='coerce')
        changed_col = mask_ruido_v2 & (
            (orig_s.isna() != inj_s.isna()) |
            ((orig_s != inj_s) & orig_s.notna() & inj_s.notna())
        )
        changed_own = changed_col & df_trabajo['sensor_anomalia'].eq(sensor)
        changed_outside = changed_col & ~df_trabajo['sensor_anomalia'].eq(sensor)
        v2n = mask_sensor & orig_s.notna() & inj_s.isna()
        n2v = mask_sensor & orig_s.isna() & inj_s.notna()
        delta = (inj_s - orig_s).where(mask_sensor)
        cambios_fuera_total += int(changed_outside.sum())
        value_to_nan_total += int(v2n.sum())
        filas.append({
            'sensor': sensor,
            'filas_ruido_sensor': int(mask_sensor.sum()),
            'cambios_columna': int(changed_col.sum()),
            'cambios_propios': int(changed_own.sum()),
            'cambios_fuera_sensor': int(changed_outside.sum()),
            'value_to_nan_sensor_propio': int(v2n.sum()),
            'nan_to_value_sensor_propio': int(n2v.sum()),
            'delta_abs_min': float(delta.abs().min()) if mask_sensor.any() else np.nan,
            'delta_abs_mediana': float(delta.abs().median()) if mask_sensor.any() else np.nan,
            'delta_abs_max': float(delta.abs().max()) if mask_sensor.any() else np.nan,
            'pct_positivo': float((delta.loc[mask_sensor] > 0).mean() * 100) if mask_sensor.any() else np.nan,
        })

    df_audit_ruido_full_v3 = pd.DataFrame(filas)
    out_audit_ruido_v2 = os.path.join(DATA_RESULTS, '02_ruido_full_v3_auditoria.csv')
    df_audit_ruido_full_v3.to_csv(out_audit_ruido_v2, index=False)
    print('\nAuditoria Ruido full_v3:')
    print(df_audit_ruido_full_v3.round(3).to_string(index=False))
    print(f'Auditoria Ruido full_v3 guardada: {out_audit_ruido_v2}')
    print(f'Cambios fuera del sensor etiquetado: {cambios_fuera_total}')
    print(f'Value->NaN en sensor propio Ruido: {value_to_nan_total}')

    if 'PRAD' in sensores_audit_ruido:
        fechas_audit = pd.to_datetime(df_trabajo['Fecha'], errors='coerce')
        mask_prad = mask_ruido_v2 & df_trabajo['sensor_anomalia'].eq('PRAD')
        prad_noche = mask_prad & ~fechas_audit.dt.hour.isin(RUIDO_V2_PRAD_HORAS)
        prad_bajo = mask_prad & (pd.to_numeric(df_original['PRAD'], errors='coerce') < RUIDO_V2_PRAD_MIN)
        print(f'PRAD Ruido nocturno: {int(prad_noche.sum())}')
        print(f'PRAD Ruido con base < {RUIDO_V2_PRAD_MIN}: {int(prad_bajo.sum())}')

    if cambios_fuera_total != 0:
        print('ALERTA: Ruido modifico sensores no etiquetados.')
    if value_to_nan_total != 0:
        print('ALERTA: Ruido creo NaNs en sensor propio.')
else:
    print('Auditoria Ruido full_v3 omitida: faltan df_trabajo o df_original.')


In [ ]:
# --- Auditoria especifica Ruido PVV full_v3 ---------------------------------
if 'df_eventos_ruido_v2' in globals() and 'df_original' in locals() and 'PVV' in df_original.columns:
    pvv_eventos = df_eventos_ruido_v2[df_eventos_ruido_v2['sensor'].eq('PVV')].copy()
    if not pvv_eventos.empty:
        pvv_orig = pd.to_numeric(df_original['PVV'], errors='coerce')
        pvv_diff_30s = pvv_orig.diff().abs()
        pvv_mask_nat = pvv_orig.notna() & pvv_orig.shift(1).notna() & (pvv_orig.shift(1) >= RUIDO_V2_PVV_MIN)
        q_nat = pvv_diff_30s[pvv_mask_nat].quantile([0.95, 0.99, 0.995, 0.999])
        print('\nAuditoria PVV Ruido full_v3:')
        print('Saltos naturales PVV 30s con base previa >= minimo:')
        print(q_nat.round(3).to_string())
        print('\nSaltos inyectados PVV full_v3:')
        print(pvv_eventos['abs_delta'].describe(percentiles=[0.05,0.25,0.5,0.75,0.95]).round(3).to_string())
        print(f"PVV eventos con base < {RUIDO_V2_PVV_MIN}: {int((pvv_eventos['valor_original'] < RUIDO_V2_PVV_MIN).sum())}")
        print(f"PVV eventos con delta < {RUIDO_V2_DELTAS['PVV']['min']}: {int((pvv_eventos['abs_delta'] < RUIDO_V2_DELTAS['PVV']['min']).sum())}")
else:
    print('Auditoria PVV Ruido full_v3 omitida: faltan df_eventos_ruido_v2 o df_original.')


## 5. Valores Fuera de Rango
Usamos los rangos fÃ­sicos reales del hardware (v3 - M3) en lugar de percentiles estadÃ­sticos.

In [ ]:
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print("Calculando estadÃ­sticas de rango para las columnas objetivo (usando df_original):\n")
    
    stats_list = []

    for col in columnas_objetivo_existentes:
        if pd.api.types.is_numeric_dtype(df_original[col]):
            # Calcular estadÃ­sticas
            min_obs = df_original[col].min()
            max_obs = df_original[col].max()
            mean_obs = df_original[col].mean()
            std_obs = df_original[col].std()
            
            # Calcular percentiles
            p0_1 = df_original[col].quantile(0.001) # 0.1-th percentile
            p1 = df_original[col].quantile(0.01)   # 1st percentile
            p5 = df_original[col].quantile(0.05)   # 5th percentile
            p95 = df_original[col].quantile(0.95)  # 95th percentile
            p99 = df_original[col].quantile(0.99)  # 99th percentile
            p99_9 = df_original[col].quantile(0.999)# 99.9-th percentile
            
            stats_list.append({
                'Columna': col,
                'Min Observado': min_obs,
                'P0.1 (0.1%)': p0_1,
                'P1 (1%)': p1,
                'P5 (5%)': p5,
                'Media': mean_obs,
                'P95 (95%)': p95,
                'P99 (99%)': p99,
                'P99.9 (99.9%)': p99_9,
                'Max Observado': max_obs,
                'Std Dev': std_obs
            })
        else:
            print(f"  Advertencia: Columna '{col}' no es numÃ©rica. Se omitirÃ¡n las estadÃ­sticas.")

    if stats_list:
        df_stats = pd.DataFrame(stats_list)
        print(df_stats.to_string()) # .to_string() para mostrar todas las filas y columnas
    else:
        print("No se pudieron calcular estadÃ­sticas para ninguna columna.")

else:
    print("Error: 'df_original' o 'columnas_objetivo_existentes' no estÃ¡n definidos o la lista de columnas estÃ¡ vacÃ­a.")

In [ ]:
# RANGOS_FISICOS viene de config.py - no hace falta redefinirlos aquÃ­.
# EstÃ¡n en config.py para que cualquier notebook del proyecto los use de forma centralizada.
# Si necesitas ajustar un rango, cÃ¡mbialo solo en config.py.
rangos_validos_sensores = RANGOS_FISICOS

print("Rangos fÃ­sicos del hardware (fuente: AGROCONNECT_Variables_PLCs.xlsx):")
for sensor, rango in rangos_validos_sensores.items():
    print(f"  {sensor:10s}: {rango['min']:8.1f} -> {rango['max']:8.1f}")


In [ ]:
def inyectar_valores_fuera_rango(df, porcentaje_filas_afectadas, rangos_sensores,
                                    columnas_con_rango, num_sensores_afectados_por_fila=1,
                                    margen_error_factor=0.1,
                                    instancias_por_sensor_temporal=None,
                                    fecha_inicio_test_final=None,
                                    random_state=42):
    """Inyecta valores fuera de rango asegurando cambio real y usando los rangos de config.py.

    v2: si se proporciona instancias_por_sensor_temporal, reparte cuotas fijas por
        sensor en train/test usando fecha_inicio_test_final. Esto garantiza cobertura
        real en el fold final para sensores interiores (XCO2I/XHINV/XTINV) que con
        inyeccion aleatoria por porcentaje caen con 0 instancias en test.
    """
    if not columnas_con_rango or not rangos_sensores:
        print('No se proporcionaron columnas o rangos.')
        return df
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()

    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None
    col_indices = {c: df_modificado.columns.get_loc(c) for c in columnas_con_rango if c in df_modificado.columns}

    # -- Modo cuotas temporales por sensor ----------------------------------------
    if instancias_por_sensor_temporal and fecha_inicio_test_final:
        if 'Fecha' not in df_modificado.columns:
            print('Aviso: no existe Fecha; no se pueden usar cuotas temporales. Se omite OOR.')
            return df_modificado

        fecha_corte = pd.Timestamp(fecha_inicio_test_final)
        fechas = pd.to_datetime(df_modificado['Fecha'])
        mask_normal = df_modificado['etiqueta_deteccion'] == 'normal'

        splits = {
            'train': df_modificado.index[mask_normal & (fechas <  fecha_corte)].to_numpy(),
            'test':  df_modificado.index[mask_normal & (fechas >= fecha_corte)].to_numpy(),
        }

        print('Modo cuotas temporales OOR por sensor.')
        print(f'  Corte test final: {fecha_corte}')

        total_inyectadas = 0
        for etiqueta_split, cuotas in instancias_por_sensor_temporal.items():
            pool = rng.permutation(splits[etiqueta_split])
            ptr = 0  # puntero compartido - cada sensor toma su slice sin solapamiento
            for sensor, n_objetivo in cuotas.items():
                if sensor not in rangos_sensores or sensor not in col_indices:
                    print(f'  Aviso: {sensor} no disponible, se omite.')
                    continue
                r = rangos_sensores[sensor]
                span = r['max'] - r['min']
                margen = span * margen_error_factor if span > 0 else abs(r['max']) * 0.1 or 1.0
                col_i = col_indices[sensor]

                inyectadas = 0
                while ptr < len(pool) and inyectadas < n_objetivo:
                    fila_idx = pool[ptr]
                    ptr += 1
                    if pd.isna(df_modificado.iat[fila_idx, col_i]):
                        continue
                    nuevo = (r['min'] - margen) if rng.random() < 0.5 else (r['max'] + margen)
                    val_actual = float(df_modificado.iat[fila_idx, col_i])
                    if np.isclose(nuevo, val_actual, rtol=0.0, atol=1e-12):
                        continue
                    df_modificado.iat[fila_idx, col_i]   = nuevo
                    df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
                    df_modificado.iat[fila_idx, tipo_idx] = 'Valores Fuera de Rango'
                    if sensor_idx is not None:
                        df_modificado.iat[fila_idx, sensor_idx] = sensor
                    inyectadas += 1
                    total_inyectadas += 1

                estado = 'OK' if inyectadas >= n_objetivo else f'AVISO: solo {inyectadas}/{n_objetivo}'
                print(f'  {sensor:10s} [{etiqueta_split:5s}]: {inyectadas:5d} instancias  {estado}')

        print(f'OOR cuotas temporales completado: {total_inyectadas:,} instancias total.')
        return df_modificado

    # -- Modo legado: porcentaje aleatorio (sin cuotas) ---------------------------
    idx_normales = df_modificado.index[df_modificado['etiqueta_deteccion'] == 'normal'].to_numpy()
    n_objetivo = int(len(idx_normales) * porcentaje_filas_afectadas)
    if n_objetivo == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado

    filas_pool = rng.permutation(idx_normales)
    print('Inyectando valores fuera de rango en hasta ' + str(n_objetivo) + ' filas...')

    filas_modificadas = 0
    count = 0

    for fila_idx in filas_pool:
        if filas_modificadas >= n_objetivo:
            break

        cols_disponibles = [
            c for c in columnas_con_rango
            if c in rangos_sensores and c in col_indices and not pd.isna(df_modificado.iat[fila_idx, col_indices[c]])
        ]
        if not cols_disponibles:
            continue

        elegidas = rng.choice(cols_disponibles, size=min(num_sensores_afectados_por_fila, len(cols_disponibles)), replace=False)
        cambios_fila = 0
        sensores_modificados_fila = []

        for col in np.atleast_1d(elegidas):
            r = rangos_sensores[col]
            span = r['max'] - r['min']
            margen = span * margen_error_factor if span > 0 else abs(r['max']) * 0.1 or 1.0
            nuevo = (r['min'] - margen) if rng.random() < 0.5 else (r['max'] + margen)
            val_actual = float(df_modificado.iat[fila_idx, col_indices[col]])

            if np.isclose(nuevo, val_actual, rtol=0.0, atol=1e-12):
                continue

            df_modificado.iat[fila_idx, col_indices[col]] = nuevo
            count += 1
            cambios_fila += 1
            sensores_modificados_fila.append(str(col))

        if cambios_fila == 0:
            continue

        df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx] = 'Valores Fuera de Rango'
        if sensor_idx is not None:
            df_modificado.iat[fila_idx, sensor_idx] = ','.join(sensores_modificados_fila)
        filas_modificadas += 1

    if filas_modificadas < n_objetivo:
        print('Aviso: solo se modificaron ' + str(filas_modificadas) + ' de ' + str(n_objetivo) + ' filas objetivo.')

    print('Inyeccion fuera de rango completada. ' + str(count) + ' puntos en ' + str(filas_modificadas) + ' filas.')
    return df_modificado


In [ ]:
# v3: rangos_validos_sensores ya contiene los rangos fisicos reales (celda anterior)
columnas_con_rango_definido = list(rangos_validos_sensores.keys())

if 'df_trabajo' not in locals() and 'df_original' in locals():
    df_trabajo = df_original.copy()

if ('df_trabajo' in locals() and
    'columnas_con_rango_definido' in locals() and columnas_con_rango_definido and
    'rangos_validos_sensores' in locals() and rangos_validos_sensores):

    cfg_oor = INYECCION['fuera_rango']
    porcentaje_filas_oor   = cfg_oor['porcentaje_filas']
    margen_factor          = cfg_oor['margen_error_factor']
    num_sensores_afectados = cfg_oor['num_sensores_por_fila']
    instancias_por_sensor_temporal = cfg_oor.get('instancias_por_sensor_temporal')
    fecha_inicio_test_final        = cfg_oor.get('fecha_inicio_test_final')

    if 'etiqueta_deteccion' not in df_trabajo.columns:
        df_trabajo['etiqueta_deteccion'] = 'normal'
    if 'etiqueta_tipo_anomalia' not in df_trabajo.columns:
        df_trabajo['etiqueta_tipo_anomalia'] = 'Ninguna'

    etiquetas_deteccion_antes_oor = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_oor = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print("\nIniciando inyeccion de 'Valores Fuera de Rango' con rangos fisicos reales (v3)...")
    df_trabajo = inyectar_valores_fuera_rango(
        df_trabajo, porcentaje_filas_oor, rangos_validos_sensores,
        columnas_con_rango_definido, num_sensores_afectados, margen_factor,
        instancias_por_sensor_temporal=instancias_por_sensor_temporal,
        fecha_inicio_test_final=fecha_inicio_test_final)

    print("\n--- Resultados inyeccion 'Valores Fuera de Rango' ---")
    print(df_trabajo['etiqueta_deteccion'].value_counts())
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))


## 6. Desviacion de Correlacion


In [ ]:
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print("Calculando la matriz de correlaciÃ³n para las columnas objetivo (usando df_original):\n")
    
    # En v6 la correlaciÃ³n debe incluir tambiÃ©n UVENT si existe, aunque no sea objetivo directo de inyecciÃ³n.
    columnas_para_corr = list(dict.fromkeys(
        list(columnas_objetivo_existentes) + [c for c in COLUMNAS_VENTILACION if c in df_original.columns]
    ))
    columnas_numericas_para_corr = df_original[columnas_para_corr].select_dtypes(include=np.number).columns
    
    if len(columnas_numericas_para_corr) < 2:
        print("No hay suficientes columnas numÃ©ricas para calcular una matriz de correlaciÃ³n significativa.")
    else:
        correlation_matrix = df_original[columnas_numericas_para_corr].corr()

        print("Matriz de CorrelaciÃ³n:")
        print(correlation_matrix)

        # Visualizar la matriz de correlaciÃ³n usando un heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
        plt.title('Matriz de CorrelaciÃ³n de Sensores y Actuadores (sobre df_original)')
        plt.show()
else:
    print("Error: 'df_original' o 'columnas_objetivo_existentes' no estÃ¡n definidos o la lista de columnas estÃ¡ vacÃ­a.")

In [ ]:
# Columnas para las que podrÃ­amos necesitar percentiles en desviaciÃ³n contextual/correlacional.
columnas_para_percentiles = list(dict.fromkeys([
    'PRAD', 'PRGINT', 'XTINV', 'XHINV', 'PTEXT', 'PHEXT', 'XCO2I',
    'UVENT_cen', 'UVENT_lN',
]))
percentiles_calculados = {}

if 'df_original' in locals():
    print("Calculando percentiles para columnas seleccionadas (desde df_original):")
    for col in columnas_para_percentiles:
        if col in df_original.columns and pd.api.types.is_numeric_dtype(df_original[col]):
            p25 = df_original[col].quantile(0.25)
            p50 = df_original[col].quantile(0.50) # Mediana
            p75 = df_original[col].quantile(0.75)
            percentiles_calculados[col] = {'p25': p25, 'p50': p50, 'p75': p75}
        else:
            print(f"  Advertencia: Columna '{col}' no es numÃ©rica o no existe. Se omitirÃ¡n sus percentiles.")
else:
    print("Error: 'df_original' no estÃ¡ definido. No se pueden calcular percentiles.")


In [ ]:
def inyectar_desviacion_correlacion(df, lista_pares, percentiles_ref, porcentaje_filas_afectadas,
                                    filtro_hora_inicio=None, filtro_hora_fin=None,
                                    instancias_por_par_temporal=None,
                                    fecha_inicio_test_final=None,
                                    random_state=42):
    """Inyecta desviacion de correlacion. Filtro temporal vectorizado (sin apply fila a fila).

    v2: si se proporciona instancias_por_par_temporal, reparte cuotas fijas por
        par en train/test usando fecha_inicio_test_final. Garantiza cobertura real
        en el fold final para pares debiles como (XCO2I, UVENT_*).
    """
    if not lista_pares or not percentiles_ref:
        print('Lista de pares o percentiles no proporcionados.')
        return df
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None

    def _intentar_inyectar(fila_idx, sA, sB):
        """Intenta inyectar DesCor para el par (sA, sB) en fila_idx. Retorna True si exito."""
        if sA not in percentiles_ref or sB not in percentiles_ref:
            return False
        if sA not in correlation_matrix.index or sB not in correlation_matrix.columns:
            return False
        vA = df_modificado.at[fila_idx, sA]
        vB = df_modificado.at[fila_idx, sB]
        if pd.isna(vA) or pd.isna(vB):
            return False
        corr = correlation_matrix.loc[sA, sB]
        if pd.isna(corr):
            return False
        pB = percentiles_ref[sB]
        pA = percentiles_ref[sA]
        nuevo = None
        if vB > pB['p75']:
            nuevo = pA['p25'] if corr > 0.3 else (pA['p75'] if corr < -0.3 else None)
        elif vB < pB['p25']:
            nuevo = pA['p75'] if corr > 0.3 else (pA['p25'] if corr < -0.3 else None)
        if nuevo is None:
            return False
        df_modificado.at[fila_idx, sA]       = nuevo
        df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx] = 'Desviacion de Correlacion'
        if sensor_idx is not None:
            df_modificado.iat[fila_idx, sensor_idx] = sA
        return True

    # -- Modo cuotas temporales por par ----------------------------------------
    if instancias_por_par_temporal and fecha_inicio_test_final:
        if 'Fecha' not in df_modificado.columns:
            print('Aviso: no existe Fecha; no se pueden usar cuotas temporales. Se omite DesCor.')
            return df_modificado

        fecha_corte = pd.Timestamp(fecha_inicio_test_final)
        fechas      = pd.to_datetime(df_modificado['Fecha'])
        mask_normal = df_modificado['etiqueta_deteccion'] == 'normal'
        if filtro_hora_inicio is not None and filtro_hora_fin is not None:
            if filtro_hora_inicio <= filtro_hora_fin:
                mask_hora = (df_modificado['Hora'] >= filtro_hora_inicio) & (df_modificado['Hora'] <= filtro_hora_fin)
            else:
                mask_hora = (df_modificado['Hora'] >= filtro_hora_inicio) | (df_modificado['Hora'] <= filtro_hora_fin)
            mask_normal = mask_normal & mask_hora

        splits = {
            'train': df_modificado.index[mask_normal & (fechas <  fecha_corte)].to_numpy(),
            'test':  df_modificado.index[mask_normal & (fechas >= fecha_corte)].to_numpy(),
        }

        print('Modo cuotas temporales DesCor por par.')
        print(f'  Corte test final: {fecha_corte}')

        total_inyectadas = 0
        for etiqueta_split, cuotas_por_par in instancias_por_par_temporal.items():
            pool      = rng.permutation(splits[etiqueta_split])
            pool_usado = set()
            ptr        = 0

            for (sA, sB), n_objetivo in cuotas_por_par.items():
                inyectadas = 0
                # Avanzar el puntero compartido buscando filas validas para este par
                i = ptr
                while i < len(pool) and inyectadas < n_objetivo:
                    fila_idx = pool[i]
                    i += 1
                    if fila_idx in pool_usado:
                        continue
                    if _intentar_inyectar(fila_idx, sA, sB):
                        pool_usado.add(fila_idx)
                        inyectadas += 1
                        total_inyectadas += 1
                ptr = i  # avanzar puntero global para el siguiente par

                estado = 'OK' if inyectadas >= n_objetivo else f'AVISO: solo {inyectadas}/{n_objetivo}'
                print(f'  ({sA:8s},{sB:10s}) [{etiqueta_split:5s}]: {inyectadas:4d}  {estado}')

        print(f'DesCor cuotas temporales completado: {total_inyectadas:,} instancias total.')
        return df_modificado

    # -- Modo legado: porcentaje aleatorio (sin cuotas) -------------------------
    mask_normal = df_modificado['etiqueta_deteccion'] == 'normal'
    if filtro_hora_inicio is not None and filtro_hora_fin is not None:
        if filtro_hora_inicio <= filtro_hora_fin:
            mask_hora = (df_modificado['Hora'] >= filtro_hora_inicio) & (df_modificado['Hora'] <= filtro_hora_fin)
        else:
            mask_hora = (df_modificado['Hora'] >= filtro_hora_inicio) | (df_modificado['Hora'] <= filtro_hora_fin)
        mask_normal = mask_normal & mask_hora
    idx_candidatos = df_modificado.index[mask_normal].to_numpy()
    n = int(len(idx_candidatos) * porcentaje_filas_afectadas)
    if n == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado
    filas = rng.choice(idx_candidatos, size=n, replace=False)
    print('Inyectando desviacion de correlacion en ' + str(n) + ' filas...')
    count = 0
    pares = list(lista_pares)
    for fila_idx in filas:
        sA, sB = pares[int(rng.integers(0, len(pares)))]
        if _intentar_inyectar(fila_idx, sA, sB):
            count += 1
    print('Inyeccion de correlacion completada. ' + str(count) + ' puntos modificados.')
    return df_modificado


In [ ]:
if ('df_trabajo' in locals() and
    'percentiles_calculados' in locals() and percentiles_calculados and
    'correlation_matrix' in locals()):

    cfg_corr_dev = INYECCION['desviacion_correlacion']
    instancias_por_par_temporal    = cfg_corr_dev.get('instancias_por_par_temporal')
    fecha_inicio_test_final_descor = cfg_corr_dev.get('fecha_inicio_test_final')

    # Pares con correlaciones fisicas robustas y percentiles bien distribuidos.
    # Se descartan pares con UVENT porque apertura=0 la mayor parte del tiempo
    # hace que p25â‰ˆp75â‰ˆ0 y la inyeccion de desviacion nunca se materializa.
    pares_para_desviacion = [
        ('PRAD',  'PRGINT'),   # radiacion ext vs. int          corrâ‰ˆ0.96
        ('XTINV', 'XHINV'),    # temp inv vs. humedad inv        corrâ‰ˆ-0.80
        ('PTEXT', 'XTINV'),    # temp ext vs. temp inv           corrâ‰ˆ0.94
        ('PHEXT', 'XHINV'),    # humedad ext vs. humedad inv     corrâ‰ˆ0.75
    ]

    etiquetas_deteccion_antes_corr = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_corr = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print('Iniciando inyeccion de Desviacion de Correlacion...')
    print(f'Pares usados: {pares_para_desviacion}')
    df_trabajo = inyectar_desviacion_correlacion(df_trabajo,
                                                pares_para_desviacion,
                                                percentiles_calculados,
                                                cfg_corr_dev['porcentaje_filas'],
                                                filtro_hora_inicio=cfg_corr_dev['filtro_hora_inicio'],
                                                filtro_hora_fin=cfg_corr_dev['filtro_hora_fin'],
                                                instancias_por_par_temporal=instancias_por_par_temporal,
                                                fecha_inicio_test_final=fecha_inicio_test_final_descor,
                                                random_state=42)

    print('Conteo etiqueta_deteccion ANTES:')
    print(etiquetas_deteccion_antes_corr)
    print('Conteo etiqueta_deteccion DESPUES:')
    print(df_trabajo['etiqueta_deteccion'].value_counts())
    print('Conteo etiqueta_tipo_anomalia DESPUES:')
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print("Variables necesarias no disponibles para inyectar DesCor.")


## 7. Anomalias Contextuales


In [ ]:
def inyectar_prad_prgint_desajuste(df, porcentaje_filas_afectadas,
                                    prad_min=200.0, ratio_normal_min=0.40,
                                    ratio_anomalo_max=0.08, random_state=42):
    """AnomalÃ­a contextual Tipo 1: PRAD alto exterior pero PRGINT anÃ³malamente bajo interior.

    Simula obstrucciÃ³n de cubierta (suciedad, sombreo inesperado).
    Es contextual porque PRGINT=10 W/m2 es normal de noche pero anÃ³malo
    cuando PRAD=600 W/m2 en el exterior.
    Solo candidatos de dÃ­a (PRAD > prad_min) y el modelo aprende la relaciÃ³n, no el horario.
    """
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()

    prad_vals   = pd.to_numeric(df_modificado['PRAD'],   errors='coerce').values
    prgint_vals = pd.to_numeric(df_modificado['PRGINT'], errors='coerce').values
    ratio_actual = np.where(prad_vals > 1.0, prgint_vals / (prad_vals + 1e-3), np.nan)

    mask = (
        (df_modificado['etiqueta_deteccion'] == 'normal') &
        (prad_vals > prad_min) &
        (ratio_actual > ratio_normal_min)
    )
    idx_candidatos = df_modificado.index[mask].to_numpy()
    if len(idx_candidatos) == 0:
        print('Sin candidatos para PRAD/PRGINT desajuste.')
        return df_modificado

    n = int(len(idx_candidatos) * porcentaje_filas_afectadas)
    if n == 0:
        print('0 filas a afectar en PRAD/PRGINT desajuste.')
        return df_modificado

    filas = rng.choice(idx_candidatos, size=n, replace=False)
    prad_col   = df_modificado.columns.get_loc('PRAD')
    prgint_col = df_modificado.columns.get_loc('PRGINT')
    det_idx    = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx   = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None

    ratios = rng.uniform(0.02, ratio_anomalo_max, size=n)
    for i, fila_idx in enumerate(filas):
        prad_val = float(df_modificado.iat[fila_idx, prad_col])
        df_modificado.iat[fila_idx, prgint_col] = prad_val * ratios[i]
        df_modificado.iat[fila_idx, det_idx]    = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx]   = 'Contextual'
        if sensor_idx is not None:
            df_modificado.iat[fila_idx, sensor_idx] = 'PRGINT'

    print(f'Inyeccion PRAD/PRGINT desajuste: {n} filas '
          f'(PRAD>{prad_min:.0f} W/m2, PRGINT->{ratio_anomalo_max*100:.0f}% del PRAD).')
    return df_modificado


def inyectar_co2_ventilacion_abierta(df, num_secuencias_afectadas,
                                      umbral_vent_abierta=40.0,
                                      prad_min_diurno=50.0,
                                      co2_anomalo_min=700, co2_anomalo_max=1200,
                                      duracion_anomalia_min=3, duracion_anomalia_max=8,
                                      secuencias_por_split=None, fecha_inicio_test_final=None,
                                      random_state=42):
    """AnomalÃ­a contextual Tipo 2: XCO2I alto cuando ventilaciÃ³n estÃ¡ abierta.

    Simula fallo de ventilaciÃ³n: UVENT dice "abierta" pero CO2 no baja.
    Es contextual porque XCO2I=900 ppm de noche (sin ventilar) es normal,
    pero es anÃ³malo cuando UVENT >= 40% y hay radiaciÃ³n diurna.
    """
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    n = len(df_modificado)

    xco2i_idx = df_modificado.columns.get_loc('XCO2I')
    det_idx   = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx  = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None

    columnas_vent = [c for c in COLUMNAS_VENTILACION if c in df_modificado.columns]
    if not columnas_vent:
        print('Sin columnas de ventilacion; inyeccion CO2 contextual omitida.')
        return df_modificado

    vent_media = df_modificado[columnas_vent].mean(axis=1, skipna=True).to_numpy(dtype=float)
    prad_vals  = pd.to_numeric(df_modificado['PRAD'], errors='coerce').to_numpy(dtype=float)

    def _construir_candidatos(mask_periodo):
        es_normal = (df_modificado['etiqueta_deteccion'] == 'normal').values & mask_periodo
        max_start = n - duracion_anomalia_max
        if max_start <= 0:
            return np.array([], dtype=int)
        cond = (
            (vent_media[:max_start] >= umbral_vent_abierta) &
            (prad_vals[:max_start]  >= prad_min_diurno) &
            ~np.isnan(vent_media[:max_start]) &
            ~np.isnan(prad_vals[:max_start])
        )
        cum   = np.concatenate([[0], np.cumsum(es_normal.astype(np.int32))])
        sumas = cum[duracion_anomalia_max:max_start + duracion_anomalia_max] - cum[:max_start]
        return np.where(cond & (sumas == duracion_anomalia_max))[0]

    def _inyectar_secuencias(candidatos, n_objetivo, etiqueta):
        es_normal_local = (df_modificado['etiqueta_deteccion'] == 'normal').values.copy()
        creadas = 0; intentos = 0; max_intentos = n_objetivo * 20
        while creadas < n_objetivo and intentos < max_intentos:
            intentos += 1
            duracion = int(rng.integers(duracion_anomalia_min, duracion_anomalia_max + 1))
            validos  = candidatos[candidatos + duracion <= n]
            validos  = validos[np.all([es_normal_local[validos + k] for k in range(duracion)], axis=0)]
            if len(validos) == 0:
                continue
            start_idx     = int(rng.choice(validos))
            co2_inyectado = float(rng.uniform(co2_anomalo_min, co2_anomalo_max))
            for k in range(duracion):
                ci = start_idx + k
                df_modificado.iat[ci, xco2i_idx] = co2_inyectado + float(rng.uniform(-10, 10))
                df_modificado.iat[ci, det_idx]    = 'anomalia'
                df_modificado.iat[ci, tipo_idx]   = 'Contextual'
                if sensor_idx is not None:
                    df_modificado.iat[ci, sensor_idx] = 'XCO2I'
            es_normal_local[start_idx:start_idx + duracion] = False
            creadas += 1
        if creadas < n_objetivo:
            print(f'  Aviso CO2 [{etiqueta}]: {creadas}/{n_objetivo} secuencias.')
        else:
            print(f'  CO2 [{etiqueta}]: {creadas} secuencias OK')
        return creadas

    if secuencias_por_split and fecha_inicio_test_final:
        fecha_corte = pd.Timestamp(fecha_inicio_test_final)
        fechas      = pd.to_datetime(df_modificado['Fecha']).values
        mask_train  = fechas < np.datetime64(fecha_corte)
        mask_test   = fechas >= np.datetime64(fecha_corte)
        print(f'Modo cuotas CO2+ventilacion por split. Corte: {fecha_corte}')
        total = 0
        for etq, n_obj in secuencias_por_split.items():
            if n_obj == 0:
                continue
            mask_p     = mask_train if etq == 'train' else mask_test
            candidatos = _construir_candidatos(mask_p)
            print(f'  [{etq}] {len(candidatos)} candidatos para {n_obj} secuencias')
            total += _inyectar_secuencias(candidatos, n_obj, etq)
        print(f'CO2+ventilacion completado: {total} secuencias total.')
        return df_modificado

    mask_todo  = np.ones(n, dtype=bool)
    candidatos = _construir_candidatos(mask_todo)
    print(f'{len(candidatos)} candidatos para {num_secuencias_afectadas} secuencias CO2+ventilacion.')
    _inyectar_secuencias(candidatos, num_secuencias_afectadas, 'global')
    return df_modificado


def inyectar_temp_inversion_contextual(df, porcentaje_filas_afectadas,
                                        prad_min=400.0, ptext_min=20.0,
                                        delta_min=3.0, delta_max=10.0,
                                        random_state=42):
    """AnomalÃ­a contextual Tipo 3: temperatura interior < exterior con alta radiaciÃ³n.

    En condiciones de PRAD > 400 W/m2 y PTEXT > 20 C, el invernadero debe
    estar mÃ¡s caliente que el exterior (efecto invernadero). XTINV < PTEXT
    es contextualmente an?malo e indica p?rdida t?rmica inesperada.
    """
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()

    prad_vals  = pd.to_numeric(df_modificado['PRAD'],  errors='coerce').values
    ptext_vals = pd.to_numeric(df_modificado['PTEXT'], errors='coerce').values
    xtinv_vals = pd.to_numeric(df_modificado['XTINV'], errors='coerce').values

    mask = (
        (df_modificado['etiqueta_deteccion'] == 'normal') &
        (prad_vals  > prad_min) &
        (ptext_vals > ptext_min) &
        (xtinv_vals >= ptext_vals)
    )
    idx_candidatos = df_modificado.index[mask].to_numpy()
    if len(idx_candidatos) == 0:
        print('Sin candidatos para temperatura inversion contextual.')
        return df_modificado

    n = int(len(idx_candidatos) * porcentaje_filas_afectadas)
    if n == 0:
        print('0 filas a afectar en temperatura inversion.')
        return df_modificado

    filas = rng.choice(idx_candidatos, size=n, replace=False)
    ptext_col = df_modificado.columns.get_loc('PTEXT')
    xtinv_col = df_modificado.columns.get_loc('XTINV')
    det_idx   = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx  = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    sensor_idx = df_modificado.columns.get_loc('sensor_anomalia') if 'sensor_anomalia' in df_modificado.columns else None

    deltas = rng.uniform(delta_min, delta_max, size=n)
    for i, fila_idx in enumerate(filas):
        ptext_val = float(df_modificado.iat[fila_idx, ptext_col])
        df_modificado.iat[fila_idx, xtinv_col] = ptext_val - deltas[i]
        df_modificado.iat[fila_idx, det_idx]   = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx]  = 'Contextual'
        if sensor_idx is not None:
            df_modificado.iat[fila_idx, sensor_idx] = 'XTINV'

    print(f'Inyeccion temp inversion contextual: {n} filas '
          f'(PRAD>{prad_min:.0f} W/m2, XTINV=PTEXT-{delta_min:.0f}~{delta_max:.0f} C).')
    return df_modificado


In [ ]:
# Aplicar inyecciÃ³n Contextual Tipo 1: PRAD/PRGINT desajuste
if 'df_trabajo' in locals():
    cfg = INYECCION['contextual_prad_prgint']
    df_trabajo = inyectar_prad_prgint_desajuste(
        df_trabajo,
        porcentaje_filas_afectadas=cfg['porcentaje_filas'],
        prad_min=cfg['prad_min'],
        ratio_normal_min=cfg['ratio_normal_min'],
        ratio_anomalo_max=cfg['ratio_anomalo_max'],
    )
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())


### 7b. CO2 Alto con Ventilacion Abierta + Temperatura Inversion

In [ ]:
# Aplicar inyecciÃ³n Contextual Tipo 2: CO2 alto con ventilaciÃ³n abierta
if 'df_trabajo' in locals():
    cfg = INYECCION['contextual_co2_vent']
    df_trabajo = inyectar_co2_ventilacion_abierta(
        df_trabajo,
        num_secuencias_afectadas=cfg['num_secuencias'],
        umbral_vent_abierta=cfg['umbral_vent_abierta'],
        prad_min_diurno=cfg['prad_min_diurno'],
        co2_anomalo_min=cfg['co2_anomalo_min'],
        co2_anomalo_max=cfg['co2_anomalo_max'],
        duracion_anomalia_min=cfg['duracion_min'],
        duracion_anomalia_max=cfg['duracion_max'],
        fecha_inicio_test_final=cfg.get('fecha_inicio_test_final'),
        secuencias_por_split=cfg.get('secuencias_por_split'),
        random_state=42,
    )
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())


In [ ]:
# Aplicar inyecciÃ³n Contextual Tipo 3: temperatura interior < exterior con alta radiaciÃ³n
if 'df_trabajo' in locals():
    cfg = INYECCION['contextual_temp_inv']
    df_trabajo = inyectar_temp_inversion_contextual(
        df_trabajo,
        porcentaje_filas_afectadas=cfg['porcentaje_filas'],
        prad_min=cfg['prad_min'],
        ptext_min=cfg['ptext_min'],
        delta_min=cfg['delta_min'],
        delta_max=cfg['delta_max'],
    )
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())


## 8. Resumen de etiquetas inyectadas

In [ ]:
if 'df_trabajo' in locals():
    print("Conteo completo de valores en 'etiqueta_tipo_anomalia':")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))

    print("\nConteo completo de valores en 'etiqueta_deteccion':")
    print(df_trabajo['etiqueta_deteccion'].value_counts(dropna=False))
    
    # Un pequeÃ±o resumen del total de filas y filas anÃ³malas
    total_filas = len(df_trabajo)
    filas_anomalas_detectadas = (df_trabajo['etiqueta_deteccion'] == 'anomalia').sum()
    print(f"\nTotal de filas en df_trabajo: {total_filas}")
    print(f"Total de filas marcadas como 'anomalia' en 'etiqueta_deteccion': {filas_anomalas_detectadas}")
    if total_filas > 0:
        print(f"Porcentaje de filas anÃ³malas: {filas_anomalas_detectadas / total_filas * 100:.2f}%")

else:
    print("Error: 'df_trabajo' no estÃ¡ definido.")

## 9. Visualizacion rapida


## 9b. Visualizacion especifica: Sensor Atascado por sensor


In [ ]:
# Visualizacion: Sensor Atascado inyectado, un subplot por sensor.
# Azul = se?al original | Rojo = SA inyectado atribuido a ESTE sensor.

# Restaurar Fecha como columna si alguna celda anterior hizo set_index.
for _df_name in ['df_original', 'df_trabajo']:
    _df = locals().get(_df_name)
    if _df is not None and isinstance(_df.index, pd.DatetimeIndex) and _df.index.name == 'Fecha':
        locals()[_df_name].reset_index(inplace=True)


def _sensor_match_plot_02(series, sensor):
    s = series.fillna('').astype(str)
    return s.eq(sensor) | s.str.split(',').apply(lambda vals: sensor in [v.strip() for v in vals])

if 'df_trabajo' in locals() and 'df_original' in locals():
    _sensores = [c for c in COLUMNAS_SENSORES if c in df_trabajo.columns]
    _n = len(_sensores)

    for _df in [df_original, df_trabajo]:
        if 'Fecha' in _df.columns:
            _df['Fecha'] = pd.to_datetime(_df['Fecha'])

    _mask_sa = df_trabajo['etiqueta_tipo_anomalia'].eq('Sensor Atascado')
    print(f"Total filas Sensor Atascado: {int(_mask_sa.sum())}")

    fig, axes = plt.subplots(_n, 1, figsize=(22, 3.5 * _n), sharex=False)
    if _n == 1:
        axes = [axes]

    for ax, sensor in zip(axes, _sensores):
        ax.plot(df_original['Fecha'], df_original[sensor],
                color='steelblue', alpha=0.5, lw=0.6, label='Original')

        if 'sensor_anomalia' in df_trabajo.columns:
            _mask_sensor = _sensor_match_plot_02(df_trabajo['sensor_anomalia'], sensor)
        else:
            _mask_sensor = pd.Series(False, index=df_trabajo.index)
        _mask_sa_sensor = _mask_sa & _mask_sensor
        _df_sa = df_trabajo.loc[_mask_sa_sensor]
        if not _df_sa.empty:
            ax.scatter(_df_sa['Fecha'], _df_sa[sensor],
                       color='red', s=12, alpha=0.8,
                       label=f'SA en {sensor} ({len(_df_sa)})', zorder=4)

        ax.set_title(f'{sensor}  ({len(_df_sa)} filas SA propias)', fontsize=10)
        ax.set_ylabel(sensor)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, linestyle='--', alpha=0.4)

    axes[-1].set_xlabel('Fecha')
    plt.suptitle('Sensor Atascado inyectado por sensor propio', fontsize=13, y=1.005)
    plt.tight_layout()
    plt.show()
else:
    print('df_original o df_trabajo no disponibles.')


In [ ]:
# Visualizacion de inyeccion por sensor.
# Rojo = anomalias inyectadas atribuidas a la columna visualizada.
# Gris = anomalias de otros sensores, solo como contexto visual.
columna = 'XTINV'

try:
    for _df_name in ['df_original', 'df_trabajo']:
        _df = locals().get(_df_name)
        if _df is not None:
            if 'Fecha' in _df.columns:
                _df['Fecha'] = pd.to_datetime(_df['Fecha'])
                _df.set_index('Fecha', inplace=True)
            elif not isinstance(_df.index, pd.DatetimeIndex):
                print(f"Advertencia: {_df_name} no tiene DatetimeIndex ni columna Fecha adecuada.")
except Exception as e:
    print(f"Error al configurar indices para DataFrames en memoria: {e}")


def _sensor_match_plot_02(series, sensor):
    s = series.fillna('').astype(str)
    return s.eq(sensor) | s.str.split(',').apply(lambda vals: sensor in [v.strip() for v in vals])

if 'df_original' in locals() and 'df_trabajo' in locals():
    fig, axes = plt.subplots(2, 1, figsize=(22, 10), sharex=True)

    if isinstance(df_original.index, pd.DatetimeIndex) and columna in df_original.columns:
        axes[0].plot(df_original.index, df_original[columna],
                     label='Original', alpha=0.7, color='blue')
        axes[0].set_title(f'{columna} - Datos originales')
        axes[0].set_ylabel(columna)
        axes[0].legend()
        axes[0].grid(True, linestyle='--', alpha=0.5)
    else:
        axes[0].set_title(f"{columna} - original no disponible o indice incorrecto")

    if (isinstance(df_trabajo.index, pd.DatetimeIndex) and columna in df_trabajo.columns
            and 'etiqueta_tipo_anomalia' in df_trabajo.columns):
        axes[1].plot(df_trabajo.index, df_trabajo[columna],
                     label='Con anomal?as', linestyle='dashed', alpha=0.7, color='orange')

        # Solo anomal?as corregibles/inyectadas. Las filas 'excluir' por NaN no
        # corregible se informan aparte para no inflar los puntos de anomal?a.
        if 'etiqueta_deteccion' in df_trabajo.columns:
            _mask_anom = df_trabajo['etiqueta_deteccion'].eq('anomalia')
            _mask_excluidas = df_trabajo['etiqueta_deteccion'].eq('excluir')
        else:
            _mask_anom = df_trabajo['etiqueta_tipo_anomalia'].notna()
            _mask_excluidas = pd.Series(False, index=df_trabajo.index)
        if 'sensor_anomalia' in df_trabajo.columns:
            _mask_sensor = _sensor_match_plot_02(df_trabajo['sensor_anomalia'], columna)
        else:
            _mask_sensor = pd.Series(False, index=df_trabajo.index)

        _mask_propias = _mask_anom & _mask_sensor
        _mask_otras = _mask_anom & ~_mask_sensor

        if _mask_otras.any():
            axes[1].scatter(df_trabajo.index[_mask_otras], df_trabajo.loc[_mask_otras, columna],
                            color='silver', label='Anomal?a en otro sensor', s=12,
                            alpha=0.18, zorder=2)
        if _mask_propias.any():
            axes[1].scatter(df_trabajo.index[_mask_propias], df_trabajo.loc[_mask_propias, columna],
                            color='red', label=f'Anomal?a en {columna}', s=35,
                            edgecolors='black', zorder=3)

        axes[1].set_title(
            f'{columna} - Datos con anomal?as | propias={int(_mask_propias.sum()):,} | globales={int(_mask_anom.sum()):,}'
        )
        axes[1].set_xlabel('Fecha')
        axes[1].set_ylabel(columna)
        axes[1].legend()
        axes[1].grid(True, linestyle='--', alpha=0.5)
    else:
        axes[1].set_title(f"{columna} - df_trabajo no disponible o columnas incompletas")

    plt.tight_layout()
    plt.show()
else:
    print('Los DataFrames df_original y/o df_trabajo no est?n disponibles en memoria.')


## 10. Guardado
Revertimos la etiqueta temporal `excluido_inyeccion` y guardamos `PARQUET_02`.


In [ ]:
# Trazabilidad real de anomalias inyectadas
# sensor_anomalia/anomalia_id son metadatos de evaluacion; se excluyen como features en config.py.
if 'df_trabajo' in locals() and 'df_original' in locals():
    for _df_name in ['df_original', 'df_trabajo']:
        _df = locals().get(_df_name)
        if isinstance(_df, pd.DataFrame) and isinstance(_df.index, pd.DatetimeIndex) and _df.index.name == 'Fecha':
            locals()[_df_name] = _df.reset_index()

    if 'sensor_anomalia' not in df_trabajo.columns:
        df_trabajo['sensor_anomalia'] = pd.NA
    if 'anomalia_id' not in df_trabajo.columns:
        df_trabajo['anomalia_id'] = pd.NA

    if len(df_original) != len(df_trabajo):
        print('Aviso trazabilidad: df_original y df_trabajo tienen longitudes distintas; no se reconstruye sensor_anomalia.')
    else:
        sensores_trace = [c for c in COLUMNAS_SENSORES if c in df_original.columns and c in df_trabajo.columns]
        idx_anom = df_trabajo.index[df_trabajo['etiqueta_deteccion'].eq('anomalia')]
        idx_sin_sensor = df_trabajo.index[
            df_trabajo['etiqueta_deteccion'].eq('anomalia') &
            df_trabajo['sensor_anomalia'].isna()
        ]

        sensores_por_fila = []
        for idx in idx_sin_sensor:
            sensores_modificados = []
            for sensor in sensores_trace:
                val_orig = df_original.at[idx, sensor]
                val_mod = df_trabajo.at[idx, sensor]
                ambos_nan = pd.isna(val_orig) and pd.isna(val_mod)
                un_nan = pd.isna(val_orig) != pd.isna(val_mod)
                if ambos_nan:
                    cambio = False
                elif un_nan:
                    cambio = True
                else:
                    cambio = not np.isclose(float(val_orig), float(val_mod), rtol=0.0, atol=1e-12)
                if cambio:
                    sensores_modificados.append(sensor)

            sensores_por_fila.append(','.join(sensores_modificados) if sensores_modificados else pd.NA)

        if len(idx_sin_sensor) > 0:
            df_trabajo.loc[idx_sin_sensor, 'sensor_anomalia'] = sensores_por_fila

        keys = df_trabajo[['etiqueta_tipo_anomalia', 'sensor_anomalia']].fillna('SIN_SENSOR')
        mask_anom = df_trabajo['etiqueta_deteccion'].eq('anomalia')
        cambio_bloque = (
            mask_anom.ne(mask_anom.shift(fill_value=False)) |
            keys.ne(keys.shift()).any(axis=1)
        )
        bloque = cambio_bloque.cumsum()
        bloques_anom = bloque[mask_anom]
        codigos = pd.factorize(bloques_anom)[0] + 1
        df_trabajo.loc[mask_anom, 'anomalia_id'] = [f'A{n:06d}' for n in codigos]

        print('Trazabilidad directa + reconstruccion de faltantes completada: sensor_anomalia + anomalia_id')
        print('Anomalias sin sensor_anomalia:', int(df_trabajo.loc[idx_anom, 'sensor_anomalia'].isna().sum()))
        print('\nSensor Atascado por sensor_anomalia:')
        mask_sa_trace = df_trabajo['etiqueta_tipo_anomalia'].eq('Sensor Atascado')
        print(df_trabajo.loc[mask_sa_trace, 'sensor_anomalia'].value_counts(dropna=False).sort_index())

        fecha_inicio_test_final = INYECCION.get('sensor_atascado', {}).get('fecha_inicio_test_final')
        if fecha_inicio_test_final and 'Fecha' in df_trabajo.columns:
            fecha_corte = pd.Timestamp(fecha_inicio_test_final)
            fechas = pd.to_datetime(df_trabajo['Fecha'])
            split_trace = np.where(fechas >= fecha_corte, 'test', 'train')
            df_trace = pd.DataFrame({
                'split': split_trace,
                'tipo': df_trabajo['etiqueta_tipo_anomalia'],
                'sensor': df_trabajo['sensor_anomalia'].fillna('sin_sensor'),
            })
            print('\nSensor Atascado por sensor_anomalia y split temporal configurado:')
            print(pd.crosstab(
                df_trace.loc[df_trace['tipo'].eq('Sensor Atascado'), 'sensor'],
                df_trace.loc[df_trace['tipo'].eq('Sensor Atascado'), 'split'],
            ).to_string())
else:
    print('Trazabilidad no reconstruida: faltan df_trabajo o df_original.')


In [ ]:
# Guardar el dataset completo: normal + anomalia + excluir.
# Las filas con usar_en_train=False (excluir) se filtran en 03; se guardan aqui para trazabilidad.
# Si la celda de visualizacion ejecuto set_index('Fecha'), restaurar Fecha como columna
if isinstance(df_trabajo.index, pd.DatetimeIndex) and df_trabajo.index.name == 'Fecha':
    df_trabajo = df_trabajo.reset_index()

# Guardar el dataset con anomalias inyectadas full_v3
df_trabajo.to_parquet(PARQUET_02, index=False)
print(f"Guardado: {PARQUET_02}")
print(f"Shape: {df_trabajo.shape}")
print("Distribucion final de etiquetas:")
print(df_trabajo['etiqueta_deteccion'].value_counts())
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())